# Forgetting vs plasticity when a spiking keyword spotter learns a new word

This notebook continues `snn_gscv2_pipeline.ipynb`. It trains the same delta-encoded 3 × 512 spiking network on Google Speech Commands v2, finds each class's neurons (MI, Fisher, and blends of the two), and then studies **how much the old classes are forgotten versus how well "zero" is learned** under every knob that trades one for the other. Everything here is trained with BPTT.

| Stage | What it does | Saved in `RUN_DIR/` |
|---|---|---|
| 0–6 | Setup, data, front end, encoder, network (as in the pipeline notebook) | `config.json`, `log.txt`, `checks.json` |
| 7 | Base training with **early stopping**: stop once validation loss has not improved for 3 epochs in a row (cosine schedule over a 100-epoch cap); keep the lowest-validation-loss network | `07_train/` |
| 8–10 | Per-class Fisher, Omega, MI and neuron groups | `08_fisher/` … `10_groups/` |
| 11 / 11b | Ablation of the groups, and of blended groups for α = 0, 0.1, …, 1 in two rank modes | `11_ablation/`, `11b_combo/` |
| 12a | **How many epochs** to learn "zero": six approaches trained 20 epochs, scored every few epochs; E\* picked on validation | `12a_epochs/` |
| 12b | Every regular approach and one masked approach per α, at E\* | `12b_arms/` |
| 12c | **How many neurons to freeze**: 5–70% of the hidden neurons, chosen by MI, Fisher (global or within-layer), the best blend, or at random | `12c_budget/` |
| 12d | **Learning rate × replay fraction**, with no mask and with the best mask | `12d_tradeoff/` |
| 12e | The key comparison repeated over 3 seeds | `12e_seeds/` |
| 12f | **Stability first**: keep old-class accuracy within 2 points, then maximise "zero": output calibration, distillation, soft freezing (plain, Fisher- or blend-weighted), 80–95% freezing | `12f_stability/` |
| 12g | The best 12f settings (plus readout-only and 25% replay) repeated over seeds, as trained and calibrated | `12g_stability_seeds/` |
| 13 | Figures and `summary.json` | `summary.json`, `plots/` |

**Selection rule.** Every choice (E\*, α\*, the best freeze budget) is made on the **validation** split; every reported number is on the held-out **test** split.

**Trade-off measures.** *Forgetting*: drift (old-class accuracy drop with the new output ignored) and 13-class forgetting (including old clips the new output steals). *Plasticity*: "zero" accuracy and AUC. *Combined*: 13-class accuracy and **hmean** = harmonic mean of old-class accuracy and "zero" accuracy (13-class decisions), the criterion used for every selection.

**How to run.** Run once with `SMOKE_TEST = True` (synthetic data, minutes), then set it to `False`. `DATA_ROOT` points at the extracted GSC v2 folder; `CACHE_DIR` reuses the audio already decoded by the pipeline notebook. Each run of stage 12 is saved as it finishes, so an interrupted run resumes where it stopped. No internet access is needed.

In [ ]:
import os, sys, json, math, time, wave, zlib, random, platform
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import matplotlib
    if "ipykernel" not in sys.modules:          # running as a plain script: draw off-screen
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_PLT = True
except Exception:
    HAVE_PLT = False

print(f"python {platform.python_version()} | torch {torch.__version__} | numpy {np.__version__} | matplotlib {HAVE_PLT}")
if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 3) and int(np.__version__.split(".")[0]) >= 2:
    print("warning: PyTorch older than 2.3 was built against NumPy 1.x and can fail with NumPy 2 "
          "(torch.from_numpy errors). Install numpy<2 or a newer PyTorch.")

In [ ]:
import os, sys, json, math, time, wave, zlib, random, platform
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import matplotlib
    if "ipykernel" not in sys.modules:          # running as a plain script: draw off-screen
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_PLT = True
except Exception:
    HAVE_PLT = False

print(f"python {platform.python_version()} | torch {torch.__version__} | numpy {np.__version__} | matplotlib {HAVE_PLT}")
if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 3) and int(np.__version__.split(".")[0]) >= 2:
    print("warning: PyTorch older than 2.3 was built against NumPy 1.x and can fail with NumPy 2 "
          "(torch.from_numpy errors). Install numpy<2 or a newer PyTorch.")

In [ ]:
# ---------------------------------------------------------------- what to run, and where
SMOKE_TEST = True     # True: tiny synthetic dataset, every stage in minutes. False: the real GSC v2.

DATA_ROOT = r"D:\codes\Anmol papers and code\speech_commands_v2"   # folder that holds yes/, no/, ..., validation_list.txt
RUN_DIR   = r"./runs/gscv2_tradeoff"                                # every output of this notebook goes here
CACHE_DIR = r"./runs/gscv2_delta_3x512/cache"   # audio decoded by the pipeline notebook (reused); None -> RUN_DIR/cache

FORCE = set()     # stages to redo from scratch, e.g. {"07_train"}; every later stage is redone too

# ---------------------------------------------------------------- every hyperparameter
CFG = {
    "seed": 0,
    "data": {
        "keywords": ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"],
        "new_word": "zero",
        "new_word_in_base_unknown": True,
        "unknown_frac": 0.10,
        "silence_frac": 0.10,
        "sample_rate": 16000,
        "clip_len": 16000,
    },
    "augment": {"enabled": True, "shift_ms": 100.0, "noise_prob": 0.8, "noise_max_vol": 0.1},
    "frontend": {"n_fft": 400, "hop": 160, "n_mels": 64, "n_mfcc": 40, "f_min": 20.0, "f_max": 8000.0,
                 "mel_floor_min": 1e-3, "floor_batches": 10, "stats_batches": 40},
    "encoder": {"theta": 0.35},
    "net": {"hidden": [512, 512, 512], "beta": 0.9, "threshold": 1.0},
    "train": {"epochs": 100,                  # a cap: early stopping normally ends training first
              "early_stop_patience": 3,       # stop after this many epochs in a row without a lower validation loss
              "batch_size": 128, "lr": None,  # None -> 5e-4 * 256 / widest layer; cosine decay over the cap
              "val_every": 1, "max_train_batches": None, "eval_batch_size": 256},
    "analysis": {"topk": 40, "ridge": 1e-2, "mi_bins": 8, "mi_shuffles": 20, "random_reps": 5,
                 "ksweep": [20, 40, 80, 160, 320], "save_full_fisher": True},
    "combo": {"alphas": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],   # score = alpha * MI rank + (1 - alpha) * Fisher rank
              "rank_modes": ["global", "within_layer"]},
    "cl": {"batch_size": 128, "lr": None,     # None -> the base learning rate
           "local_lr": 0.1, "slda_shrinkage": 1e-2, "k_new_neurons": 64, "init_batches": 20,
           "epochs": 5, "ff": {"kappa": 4.0, "hybrid_block": 10}},    # (epochs / ff: used only by helper code kept from the pipeline)
    "tradeoff": {
        "epoch_sweep": {"max_epochs": 20, "eval_epochs": [1, 2, 3, 5, 8, 12, 16, 20],
                        "arms": ["no_mask", "mi_mask", "fisher_mask", "combo_layer_a050", "no_mask_replay", "readout_only"]},
        "budgets": [0.05, 0.10, 0.20, 0.30, 0.50, 0.70],        # share of all hidden neurons frozen
        "budget_methods": ["mi", "fisher_global", "fisher_layer", "blend_best", "random"],
        "lr_mults": [0.25, 1.0, 4.0],                            # x the continual-learning learning rate
        "replay_fracs": [0.0, 0.01, 0.05, 0.25, 0.5],            # share of each batch that is LABELLED old clips (0.5 = all old clips)
        "seeds": [0, 1, 2],
    },
    "stability": {                                       # stage 12f: stability first
        "max_old_drop_pts": 2.0,                         # old-class accuracy may fall at most this much (validation)
        "lwf_lambdas": [0.3, 1.0, 3.0, 10.0, 30.0],      # distillation strength
        "lwf_T": 2.0,                                    # distillation temperature
        "soft_importance": ["uniform", "fisher", "blend"],
        "soft_lambdas": [10.0, 100.0, 1000.0, 10000.0, 100000.0],   # soft-freezing strength (on relative weight change)
        "extreme_methods": ["blend_best", "random"],
        "extreme_budgets": [0.8, 0.9, 0.95],
    },
    "stability_seeds": {                                 # stage 12g: seeds for the stability-first winners
        "top_k": 4,                                      # best 12f settings within the limit (validation)
        "seeds": [1, 2],                                 # seed 0 is the 12f run itself
        "references": ["ref_readout_only", "ref_replay_25"],
    },
}

if SMOKE_TEST:
    RUN_DIR = "./runs/smoke_tradeoff"
    DATA_ROOT = str(Path(RUN_DIR) / "synthetic_gsc")
    CACHE_DIR = None
    CFG["net"]["hidden"] = [64, 64, 64]
    CFG["train"].update(epochs=30, batch_size=32, eval_batch_size=64)
    CFG["frontend"].update(floor_batches=3, stats_batches=4)
    CFG["analysis"].update(topk=8, mi_shuffles=5, random_reps=2, ksweep=[4, 8, 16, 32])
    CFG["cl"].update(batch_size=32, k_new_neurons=8, init_batches=3)
    CFG["tradeoff"]["epoch_sweep"].update(max_epochs=6, eval_epochs=[1, 2, 3, 4, 6])
    CFG["tradeoff"].update(seeds=[0, 1], budgets=[0.1, 0.5], lr_mults=[1.0, 4.0], replay_fracs=[0.0, 0.5])
    CFG["combo"]["alphas"] = [0.0, 0.5, 1.0]
    CFG["stability"].update(lwf_lambdas=[1.0, 10.0], soft_lambdas=[10.0, 1000.0], extreme_budgets=[0.9])
    CFG["stability_seeds"]["seeds"] = [1]

if CFG["train"]["lr"] is None:
    CFG["train"]["lr"] = 5e-4 * 256 / max(CFG["net"]["hidden"])
if CFG["cl"]["lr"] is None:
    CFG["cl"]["lr"] = CFG["train"]["lr"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN = Path(RUN_DIR)
RUN.mkdir(parents=True, exist_ok=True)
CACHE = Path(CACHE_DIR) if CACHE_DIR else RUN / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "| run dir:", RUN.resolve(), "| cache:", CACHE.resolve(), "| lr:", CFG["train"]["lr"])
if DEVICE == "cpu" and not SMOKE_TEST:
    print("warning: PyTorch sees no CUDA GPU, so the real run will train on the CPU and take many hours.")


In [ ]:
# ---------------------------------------------------------------- small helpers used by every stage
(RUN / "plots").mkdir(exist_ok=True)
LOG_FILE = RUN / "log.txt"
CHECKS_FILE = RUN / "checks.json"
CHECKS = json.loads(CHECKS_FILE.read_text(encoding="utf-8")) if CHECKS_FILE.exists() else {}
CFG_HASH = format(zlib.crc32(json.dumps(CFG, sort_keys=True).encode()), "08x")


def log(*parts):
    """Print a line and append it to RUN_DIR/log.txt, so a run can be read afterwards."""
    msg = " ".join(str(p) for p in parts)
    print(msg)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S  ") + msg + "\n")


def jsonable(x):
    """numpy / torch / Path -> plain Python, so anything can be written to JSON."""
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    if isinstance(x, torch.Tensor):
        return jsonable(x.detach().cpu().tolist())
    if isinstance(x, np.ndarray):
        return jsonable(x.tolist())
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        x = float(x)
    if isinstance(x, float) and (math.isnan(x) or math.isinf(x)):
        return None
    if isinstance(x, Path):
        return str(x)
    return x


def replace_file(tmp, path, tries=20):
    """Move tmp over path in one step, so an interrupted save never leaves a half-written file.
    On Windows a virus scanner or indexer can hold a fresh file for a moment, so retry briefly."""
    for i in range(tries):
        try:
            os.replace(tmp, path)
            return
        except PermissionError:
            if i == tries - 1:
                raise
            time.sleep(0.25)


def save_json(obj, path):
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(jsonable(obj), indent=1), encoding="utf-8")
    replace_file(tmp, path)


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def save_pt(obj, path):
    tmp = str(path) + ".tmp"
    torch.save(obj, tmp)
    replace_file(tmp, str(path))       # stopping the kernel mid-save cannot corrupt last.pt


def load_pt(path):
    try:
        return torch.load(str(path), map_location="cpu", weights_only=False)
    except TypeError:                       # torch < 1.13 has no weights_only argument
        return torch.load(str(path), map_location="cpu")


def check(name, passed, detail="", real_data_only=False):
    """Record one debug check as PASS/FAIL in RUN_DIR/checks.json. Never stops the notebook.
    real_data_only=True: the check depends on dataset size, so the smoke test only reports it (SKIP)."""
    if real_data_only and SMOKE_TEST:
        CHECKS[name] = {"passed": True, "skipped": True, "detail": str(detail)}
        save_json(CHECKS, CHECKS_FILE)
        log("  [SKIP] " + name + " (only meaningful on the real dataset)" + (": " + str(detail) if str(detail) else ""))
        return True
    CHECKS[name] = {"passed": bool(passed), "detail": str(detail)}
    save_json(CHECKS, CHECKS_FILE)
    log(("  [PASS] " if passed else "  [FAIL] ") + name + (": " + str(detail) if str(detail) else ""))
    return bool(passed)


def stage_dir(name):
    d = RUN / name
    d.mkdir(parents=True, exist_ok=True)
    return d


# Which stages can be loaded instead of recomputed. A stage's saved outputs are trusted only if its DONE.json
# exists, the CFG sections it (or any earlier stage) depends on are unchanged, and no earlier stage finished
# after it. The first stage that fails this, and every stage after it, is redone.
STAGES = ["01_index", "02_cache", "04_frontend", "05_encoder", "06_model", "07_train",
          "08_fisher", "09_mi", "10_groups", "11_ablation", "11b_combo",
          "12a_epochs", "12b_arms", "12c_budget", "12d_tradeoff", "12e_seeds", "12f_stability",
          "12g_stability_seeds"]
STAGE_CFG = {"01_index": ["seed", "data"], "04_frontend": ["augment", "frontend"], "05_encoder": ["encoder"],
             "06_model": ["net"], "07_train": ["train"], "08_fisher": ["analysis"], "11b_combo": ["combo"],
             "12a_epochs": ["cl", "tradeoff"], "12f_stability": ["stability"],
             "12g_stability_seeds": ["stability_seeds"]}
DONE_THIS_RUN = set()


def stage_hash(name):
    """Fingerprint of every CFG section this stage or an earlier one depends on."""
    keys = [k for s in STAGES[:STAGES.index(name) + 1] for k in STAGE_CFG.get(s, [])]
    return format(zlib.crc32(json.dumps({k: CFG[k] for k in keys}, sort_keys=True).encode()), "08x")


def done_time(name):
    f = RUN / name / "DONE.json"
    return load_json(f).get("t", 0.0) if f.exists() else 0.0


def stage_basis(name):
    """What a stage's partial results (last.pt, finished stage-12 arms) were built on: its settings and when
    the stages before it finished. Partial results with another basis are never reused."""
    return f"{stage_hash(name)}@{max([done_time(s) for s in STAGES[:STAGES.index(name)]], default=0.0):.3f}"


def first_stage_to_redo():
    prev = 0.0
    for s in STAGES:
        f = RUN / s / "DONE.json"
        if s in FORCE:
            return s, "it is in FORCE"
        if not f.exists():
            return s, "it has not finished yet"
        info = load_json(f)
        if info.get("stage_hash") != stage_hash(s):
            return s, "its settings changed"
        if info.get("t", 0.0) < prev:
            return s, "an earlier stage was recomputed after it"
        prev = info.get("t", 0.0)
    return None, ""


def is_done(name):
    """True if this stage's saved outputs are valid, or it was finished earlier in this run."""
    return name in DONE_THIS_RUN or (name not in REDO and (RUN / name / "DONE.json").exists())


def mark_done(name, **info):
    save_json({"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "t": time.time(), "stage_hash": stage_hash(name),
               "cfg_hash": CFG_HASH, **info}, RUN / name / "DONE.json")
    DONE_THIS_RUN.add(name)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def savefig(name):
    """Save the current figure to RUN_DIR/plots/<name>.png and show it."""
    if not HAVE_PLT:
        return
    plt.tight_layout()
    plt.savefig(RUN / "plots" / f"{name}.png", dpi=110)
    plt.show()
    plt.close("all")


seed_all(CFG["seed"])
save_json({"cfg": CFG, "cfg_hash": CFG_HASH, "smoke_test": SMOKE_TEST, "data_root": DATA_ROOT,
           "device": DEVICE, "torch": torch.__version__, "python": platform.python_version()},
          RUN / "config.json")
log("=" * 72)
log("run started |", "SMOKE TEST (synthetic data)" if SMOKE_TEST else "Google Speech Commands v2",
    "| cfg", CFG_HASH, "| device", DEVICE)

bad_force = set(FORCE) - set(STAGES)
if bad_force:
    raise ValueError(f"FORCE has unknown stage names {sorted(bad_force)}; the stages are {STAGES}")
FIRST_REDO, why = first_stage_to_redo()
REDO = set(STAGES[STAGES.index(FIRST_REDO):]) if FIRST_REDO else set()
for s in REDO:                                   # their old DONE markers and checks no longer apply
    (RUN / s / "DONE.json").unlink(missing_ok=True)
for k in [k for k in CHECKS if k[:2] in {s[:2] for s in REDO}]:
    del CHECKS[k]
save_json(CHECKS, CHECKS_FILE)
log("stages to compute: " + (f"{FIRST_REDO} ({why}) and every stage after it" if REDO else
                             "none; every stage loads its saved outputs"))

### 0b · Synthetic dataset (smoke test only)

When `SMOKE_TEST = True`, this writes a miniature fake GSC folder with the real layout: word folders, `_background_noise_/`, `validation_list.txt` and `testing_list.txt`. Every later stage then runs unchanged. Each "word" is a voiced tone complex with its own two formants, so a small network can learn it. The accuracy numbers mean nothing; only the plumbing is being tested.

In [ ]:
def write_wav(path, x, sr=16000):
    """Float array in [-1, 1] -> 16-bit PCM mono wav, the GSC format."""
    pcm = (np.clip(x, -1.0, 1.0) * 32767).astype("<i2")
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(pcm.tobytes())


def make_synthetic_gsc(root, n_per_word=60, sr=16000, seed=0):
    """A fake GSC v2 folder: keywords + the new word + 5 'other' words, ~12% validation, ~12% test."""
    root = Path(root)
    if (root / "testing_list.txt").exists():
        log("synthetic dataset already at", root)
        return
    rng = np.random.default_rng(seed)
    words = CFG["data"]["keywords"] + [CFG["data"]["new_word"]] + ["bed", "bird", "cat", "dog", "house"]
    t = np.arange(sr) / sr
    val, test = [], []
    for wi, w in enumerate(words):
        (root / w).mkdir(parents=True, exist_ok=True)
        f1 = 300.0 + 110.0 * wi                              # first formant, unique per word
        f2 = 1000.0 + 160.0 * ((wi * 7) % len(words))        # second formant
        for i in range(n_per_word):
            name = f"{int(rng.integers(0, 2**31)):08x}_nohash_{i % 3}.wav"
            dur, start = rng.uniform(0.35, 0.7), rng.uniform(0.05, 0.25)
            ramp = np.clip((t - start) / dur, 0.0, 1.0)
            env = np.sin(np.pi * ramp) ** 2 * ((t >= start) & (t < start + dur))
            f0 = rng.uniform(100.0, 160.0) * (1.0 + 0.05 * np.sin(2 * np.pi * 3.0 * t))
            phase = 2 * np.pi * np.cumsum(f0) / sr
            sig = np.zeros(sr)
            for h in range(1, 30):
                amp = np.exp(-((h * f0 - f1) / 150.0) ** 2) + 0.7 * np.exp(-((h * f0 - f2) / 250.0) ** 2)
                sig += amp * np.sin(h * phase)
            x = 0.3 * env * sig / (np.abs(sig).max() + 1e-9) + 0.002 * rng.standard_normal(sr)
            n = int(sr * rng.uniform(0.8, 1.0)) if i % 7 == 0 else sr      # a few short clips, as in GSC
            write_wav(root / w / name, x[:n], sr)
            r = rng.random()
            if r < 0.12:
                val.append(f"{w}/{name}")
            elif r < 0.24:
                test.append(f"{w}/{name}")
    bg = root / "_background_noise_"
    bg.mkdir(exist_ok=True)
    n_bg = sr * 20
    white = rng.standard_normal(n_bg)
    pink = np.cumsum(rng.standard_normal(n_bg))
    pink = pink - np.convolve(pink, np.ones(801) / 801, mode="same")
    hum = sum(np.sin(2 * np.pi * 50 * k * np.arange(n_bg) / sr) / k for k in range(1, 6)) + 0.3 * rng.standard_normal(n_bg)
    for nm, x in [("white_noise", white), ("pink_noise", pink), ("hum", hum)]:
        write_wav(bg / f"{nm}.wav", 0.5 * x / np.abs(x).max(), sr)
    (root / "validation_list.txt").write_text("\n".join(val) + "\n", encoding="utf-8")
    (root / "testing_list.txt").write_text("\n".join(test) + "\n", encoding="utf-8")
    log(f"synthetic GSC-like dataset written to {root}: {len(words)} words x {n_per_word} clips")


if SMOKE_TEST:
    make_synthetic_gsc(DATA_ROOT)

## 1 · Index the dataset

The 12 classes follow the standard GSC benchmark, with labels in this order:

- **0–9**: the ten keywords (yes, no, up, down, left, right, on, off, stop, go).
- **10 `_unknown_`**: a fixed 10% subsample of the other words. It is drawn after routing by the official lists, so no speaker crosses splits.
- **11 `_silence_`**: 10% as many clips, generated from the background-noise recordings in stage 3.
- **12**: the new word ("zero"). It is held apart from the 12 base classes and only used in stage 12. It never appears in validation or test `_unknown_`, so every evaluation set is identical before and after adding it. With `new_word_in_base_unknown = True`, training `_unknown_` may contain "zero", as in the original project.

The split comes from `validation_list.txt` and `testing_list.txt`; everything else is training. The subsample uses a fixed seed (crc32 of the split name), so it is identical on every run and machine. The original code used Python's `hash()`, which changes every run.

**Check:** the class-count table, split hygiene, and whether the per-word validation and test counts match GSC v2 exactly (e.g. 397 "yes" in validation, 419 in test).

In [ ]:
WORDS = list(CFG["data"]["keywords"])
NEW_WORD = CFG["data"]["new_word"]
UNKNOWN, SILENCE, NEW = len(WORDS), len(WORDS) + 1, len(WORDS) + 2      # labels 10, 11, 12
CLASSES_12 = WORDS + ["_unknown_", "_silence_"]
CLASSES_13 = CLASSES_12 + [NEW_WORD]
SPLITS = ["training", "validation", "testing"]
SR = CFG["data"]["sample_rate"]
CLIP = CFG["data"]["clip_len"]


def find_gsc_root(start):
    """The folder that directly holds yes/, no/, ... and testing_list.txt (searched two levels deep)."""
    start = Path(start)
    if not start.is_dir():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {start}")

    def is_root(c):
        return all((c / w).is_dir() for w in WORDS[:3]) and (c / "testing_list.txt").exists()

    def subdirs(d):
        try:
            return sorted(p for p in d.iterdir() if p.is_dir())
        except (PermissionError, OSError):     # e.g. "System Volume Information" on a Windows drive root
            return []

    if is_root(start):
        return start
    for c in subdirs(start):
        if is_root(c):
            return c
        for c2 in subdirs(c):
            if is_root(c2):
                return c2
    hint = ""
    if any(start.glob("*.tar.gz")) or any(start.glob("*/*.tar.gz")):
        hint = " (found a .tar.gz there: extract it first, e.g. `tar -xzf speech_commands_v0.02.tar.gz -C <folder>`)"
    raise FileNotFoundError(f"no Speech Commands folder (with {WORDS[:3]} and testing_list.txt) under {start}{hint}")


def read_split_list(path):
    with open(path, encoding="utf-8") as f:
        return {ln.strip().replace("\\", "/") for ln in f if ln.strip()}


def build_index(root, seed):
    """File lists per split: keyword clips with labels, the _unknown_ subsample, the new-word clips,
    and how many _silence_ clips to generate."""
    val = read_split_list(root / "validation_list.txt")
    test = read_split_list(root / "testing_list.txt")

    def split_of(rel):
        return "validation" if rel in val else "testing" if rel in test else "training"

    folders = sorted(p.name for p in root.iterdir() if p.is_dir() and not p.name.startswith("_"))
    missing = [w for w in WORDS + [NEW_WORD] if w not in folders]
    if missing:
        raise FileNotFoundError(f"missing word folders: {missing}")
    others = [d for d in folders if d not in WORDS and d != NEW_WORD]

    def wav_files(folder):
        return sorted(f for f in os.listdir(root / folder) if f.endswith(".wav"))

    files = {s: {"words": [], "labels": [], "new": [], "others": []} for s in SPLITS}
    for label, w in enumerate(WORDS):
        for f in wav_files(w):
            s = split_of(f"{w}/{f}")
            files[s]["words"].append(f"{w}/{f}")
            files[s]["labels"].append(label)
    for f in wav_files(NEW_WORD):
        files[split_of(f"{NEW_WORD}/{f}")]["new"].append(f"{NEW_WORD}/{f}")
    for d in others:
        for f in wav_files(d):
            files[split_of(f"{d}/{f}")]["others"].append(f"{d}/{f}")

    index = {"root": str(root), "other_words": others, "splits": {}}
    for s in SPLITS:
        n_words = len(files[s]["words"])
        pool = list(files[s]["others"])
        if s == "training" and CFG["data"]["new_word_in_base_unknown"]:
            pool += files[s]["new"]          # the base network may hear the new word filed under _unknown_
        pool.sort()
        n_unknown = min(len(pool), int(round(n_words * CFG["data"]["unknown_frac"])))
        rng = np.random.default_rng(seed * 7919 + zlib.crc32(s.encode()))
        pick = sorted(rng.choice(len(pool), size=n_unknown, replace=False).tolist())
        index["splits"][s] = {
            "words": files[s]["words"],
            "labels": files[s]["labels"],
            "unknown": [pool[i] for i in pick],
            "new": files[s]["new"],
            "n_silence": int(round(n_words * CFG["data"]["silence_frac"])),
            "n_other_pool": len(pool),
        }
    return index


d1 = stage_dir("01_index")
GSC_ROOT = find_gsc_root(DATA_ROOT)       # looked up on every run, so a moved dataset is still found
if is_done("01_index"):
    INDEX = load_json(d1 / "index.json")
    log("01_index: loaded", d1 / "index.json", "| dataset root", GSC_ROOT)
else:
    log("01_index: dataset root", GSC_ROOT)
    INDEX = build_index(GSC_ROOT, CFG["seed"])
    save_json(INDEX, d1 / "index.json")
    mark_done("01_index")

In [ ]:
# ---- CHECK 1: class counts per split, split hygiene, and "is this really GSC v2?"
EXPECTED_V2 = {
    "validation": {"yes": 397, "no": 406, "up": 350, "down": 377, "left": 352, "right": 363,
                   "on": 363, "off": 373, "stop": 350, "go": 372},
    "testing": {"yes": 419, "no": 405, "up": 425, "down": 406, "left": 412, "right": 396,
                "on": 396, "off": 402, "stop": 411, "go": 402},
}
COUNTS = {}
for s in SPLITS:
    sp = INDEX["splits"][s]
    c = {k: 0 for k in CLASSES_13}
    for lab in sp["labels"]:
        c[WORDS[lab]] += 1
    c["_unknown_"], c["_silence_"], c[NEW_WORD] = len(sp["unknown"]), sp["n_silence"], len(sp["new"])
    COUNTS[s] = c
print(f"{'class':<12}" + "".join(f"{s:>12}" for s in SPLITS))
for k in CLASSES_13:
    print(f"{k:<12}" + "".join(f"{COUNTS[s][k]:>12}" for s in SPLITS))
save_json(COUNTS, d1 / "class_counts.json")

files_of = {s: set(INDEX["splits"][s]["words"]) | set(INDEX["splits"][s]["unknown"]) | set(INDEX["splits"][s]["new"])
            for s in SPLITS}
overlap = (len(files_of["training"] & files_of["validation"]) + len(files_of["training"] & files_of["testing"])
           + len(files_of["validation"] & files_of["testing"]))
check("01 splits are disjoint", overlap == 0, f"{overlap} files in two splits")
check("01 every keyword present in every split", all(COUNTS[s][w] > 0 for s in SPLITS for w in WORDS))
unk_has_kw = sum(r.split("/")[0] in WORDS for s in SPLITS for r in INDEX["splits"][s]["unknown"])
check("01 _unknown_ holds no keyword clips", unk_has_kw == 0, f"{unk_has_kw} keyword clips in _unknown_")
eval_new_in_unk = sum(r.startswith(NEW_WORD + "/") for s in ["validation", "testing"] for r in INDEX["splits"][s]["unknown"])
check(f"01 '{NEW_WORD}' never in validation/test _unknown_", eval_new_in_unk == 0)
n_new_in_train_unk = sum(r.startswith(NEW_WORD + "/") for r in INDEX["splits"]["training"]["unknown"])
log(f"  '{NEW_WORD}' clips inside training _unknown_: {n_new_in_train_unk} "
    f"(new_word_in_base_unknown={CFG['data']['new_word_in_base_unknown']})")
log(f"  other words used for _unknown_: {len(INDEX['other_words'])} ({', '.join(INDEX['other_words'][:8])}, ...)")
if not SMOKE_TEST:
    mism = {s: {w: [COUNTS[s][w], EXPECTED_V2[s][w]] for w in WORDS if COUNTS[s][w] != EXPECTED_V2[s][w]}
            for s in EXPECTED_V2}
    n_mism = sum(len(v) for v in mism.values())
    check("01 per-word validation/test counts match GSC v2", n_mism == 0,
          "exact match" if n_mism == 0 else f"[found, expected] {mism}")

## 2 · Decode the audio once

Every clip is read by a small NumPy WAV parser (GSC is 16-bit PCM, 16 kHz, mono; the parser also accepts other PCM widths, float and extensible headers). It is right-padded with zeros to exactly 1 s and stored as int16, which is lossless. Each list is saved as `cache/<split>_<part>_<fingerprint>.npy`, keyed by a fingerprint of its file list. The first run decodes about 45k files, which takes a few minutes; later runs load them in seconds. GSC v2 needs about 1.5 GB of disk and RAM for this.

**Check:** array shapes, how many clips were shorter than 1 s, that no clip is all zeros, and that the background-noise recordings were found.

In [ ]:
def read_wav(path, n=CLIP, sr=SR):
    """WAV file -> (int16 array, original length), parsed with NumPy (no audio library needed).
    Handles PCM 8/16/24/32-bit, 32-bit float and WAVE_FORMAT_EXTENSIBLE headers (which Python 3.10's
    `wave` module cannot read). The array has exactly n samples, zero-padded at the end; n=None keeps
    the full length (for the long noise recordings). Other sample rates are resampled linearly."""
    with open(path, "rb") as f:
        data = f.read()
    if data[:4] != b"RIFF" or data[8:12] != b"WAVE":
        raise ValueError(f"{path}: not a RIFF/WAVE file")
    pos, fmt, pcm = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], int.from_bytes(data[pos + 4:pos + 8], "little")
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch = int.from_bytes(body[0:2], "little"), int.from_bytes(body[2:4], "little")
            rate, bits = int.from_bytes(body[4:8], "little"), int.from_bytes(body[14:16], "little")
            if tag == 0xFFFE and len(body) >= 26:            # extensible: the real format sits in the sub-format GUID
                tag = int.from_bytes(body[24:26], "little")
            fmt = (tag, ch, rate, bits)
        elif cid == b"data":
            pcm = body
        pos += 8 + size + (size & 1)                          # chunks are word-aligned
    if fmt is None or pcm is None:
        raise ValueError(f"{path}: no fmt/data chunk")
    tag, ch, rate, bits = fmt
    if ch < 1 or rate < 1 or bits not in (8, 16, 24, 32):
        raise ValueError(f"{path}: bad fmt chunk (channels {ch}, rate {rate}, bits {bits})")
    width = bits // 8
    pcm = pcm[:len(pcm) // (width * ch) * width * ch]
    if tag == 1 and bits == 16:
        x = np.frombuffer(pcm, "<i2").astype(np.float32)
    elif tag == 1 and bits == 8:
        x = (np.frombuffer(pcm, np.uint8).astype(np.float32) - 128.0) * 256.0
    elif tag == 1 and bits == 24:
        b = np.frombuffer(pcm, np.uint8).reshape(-1, 3).astype(np.int32)
        x = ((b[:, 0] | (b[:, 1] << 8) | (b[:, 2] << 16)) << 8 >> 8).astype(np.float32) / 256.0
    elif tag == 1 and bits == 32:
        x = np.frombuffer(pcm, "<i4").astype(np.float32) / 65536.0
    elif tag == 3 and bits == 32:
        x = np.frombuffer(pcm, "<f4").astype(np.float32) * 32768.0
    else:
        raise ValueError(f"{path}: unsupported WAV format (tag {tag}, {bits}-bit)")
    if ch > 1:
        x = x.reshape(-1, ch).mean(axis=1)
    if rate != sr:                              # linear resampling; GSC is already 16 kHz
        m = int(round(len(x) * sr / rate))
        x = np.interp(np.arange(m) * rate / sr, np.arange(len(x)), x)
    length = len(x)
    if n is not None:
        out = np.zeros(n, np.float32)
        out[:min(n, length)] = x[:n]
        x = out
    return np.clip(np.round(x), -32768, 32767).astype(np.int16), length


def list_key(files):
    """Short fingerprint of a file list (and clip format): the cache is rebuilt if either changes."""
    return format(zlib.crc32((f"{SR} {CLIP}\n" + "\n".join(files)).encode()), "08x")


UNREADABLE = []      # clips that could not be decoded (kept as all-zero rows so labels stay aligned)


def cached_wavs(files, tag):
    """Decode a list of clips once into [n, CLIP] int16; later runs load the .npy."""
    path = CACHE / f"{tag}_{list_key(files)}.npy"
    bad_path = path.with_name(path.stem + "_unreadable.json")
    if path.exists():
        if bad_path.exists():
            UNREADABLE.extend(load_json(bad_path))
        return np.load(path)
    t0, short, bad = time.time(), 0, []
    arr = np.zeros((len(files), CLIP), np.int16)
    for i, rel in enumerate(files):
        try:
            arr[i], length = read_wav(GSC_ROOT / rel)
            short += int(length < CLIP)
        except (ValueError, OSError) as e:
            bad.append(rel)
            if len(bad) <= 5:
                log(f"    warning: cannot decode {rel} ({e}); it stays an all-zero clip")
        if (i + 1) % 5000 == 0:
            log(f"    {tag}: {i + 1}/{len(files)} clips ({(i + 1) / (time.time() - t0):.0f} clips/s)")
    tmp = path.with_name(path.stem + "_tmp.npy")
    np.save(tmp, arr)
    replace_file(tmp, path)
    if bad:
        save_json(bad, bad_path)
        UNREADABLE.extend(bad)
    log(f"  cached {tag}: {len(files)} clips, {short} shorter than 1 s, {len(bad)} unreadable, "
        f"{arr.nbytes / 2**20:.0f} MB, {time.time() - t0:.0f} s")
    return arr


def noise_recordings():
    folder = GSC_ROOT / "_background_noise_"
    files = sorted(f for f in os.listdir(folder) if f.endswith(".wav")) if folder.is_dir() else []
    out = {}
    for f in files:
        try:
            out[f] = read_wav(folder / f, n=None)[0]
        except ValueError as e:
            log(f"  warning: skipping background-noise file {f}: {e}")
    return out


d2 = stage_dir("02_cache")
WAVS = {}
for s in SPLITS:
    sp = INDEX["splits"][s]
    WAVS[s] = {"words": cached_wavs(sp["words"], f"{s}_words"),
               "unknown": cached_wavs(sp["unknown"], f"{s}_unknown"),
               "new": cached_wavs(sp["new"], f"{s}_new")}
NOISE = noise_recordings()
if not any(len(r) > CLIP for r in NOISE.values()):
    raise FileNotFoundError(f"no background-noise recording longer than 1 s in {GSC_ROOT / '_background_noise_'}; "
                            "they are needed for the _silence_ clips and for augmentation")
sizes = {s: {k: int(v.shape[0]) for k, v in WAVS[s].items()} for s in SPLITS}
if not is_done("02_cache"):
    mark_done("02_cache", clips=sizes, noise_files=list(NOISE))
log("02_cache: ready |", sizes, "| noise files:", list(NOISE))

In [ ]:
# ---- CHECK 2: what is in the cache
for s in SPLITS:
    for part, arr in WAVS[s].items():
        if len(arr) == 0:
            continue
        rms = np.concatenate([np.sqrt(np.mean(np.square(arr[i:i + 2048], dtype=np.float32), 1)) / 32768.0
                              for i in range(0, len(arr), 2048)])          # in chunks: no GB-sized float copy
        zero_frac = float((rms == 0).mean())
        print(f"{s:<11} {part:<8} {arr.shape}  rms median {np.median(rms):.4f}  all-zero clips {zero_frac:.2%}")
ok_shape = all(a.dtype == np.int16 and a.ndim == 2 and a.shape[1] == CLIP for s in SPLITS for a in WAVS[s].values())
check("02 cache arrays are [n, 16000] int16", ok_shape)
check("02 every clip could be decoded", len(UNREADABLE) == 0,
      "all decoded" if not UNREADABLE else f"{len(UNREADABLE)} unreadable, kept as silent clips: {UNREADABLE[:5]}")
zero_all = max(float((~a.any(axis=1)).mean()) if len(a) else 0.0 for s in SPLITS for a in WAVS[s].values())
check("02 almost no all-zero clips", zero_all < 0.01, f"worst part {zero_all:.2%}")
long_noise = [f for f, r in NOISE.items() if len(r) > CLIP]
check("02 background-noise recordings found (needed for _silence_)", len(long_noise) > 0,
      f"{len(long_noise)} files, {sum(len(r) for r in NOISE.values()) / SR:.0f} s in total")

if HAVE_PLT:
    fig, axes = plt.subplots(1, 4, figsize=(14, 2.2))
    labels = np.array(INDEX["splits"]["validation"]["labels"])
    for ax, w in zip(axes, WORDS[:4]):
        i = int(np.argmax(labels == WORDS.index(w)))
        ax.plot(WAVS["validation"]["words"][i] / 32768.0, lw=0.5)
        ax.set_title(f"validation '{w}'")
        ax.set_ylim(-1, 1)
    savefig("02_waveforms")

## 3 · Batches, silence and augmentation

- **`ClipSet`** holds one split in RAM: the real clips as int16, plus a count of `_silence_` clips. Batches are cut from the arrays by index, with no DataLoader workers, so it also runs fast on Windows.
- **`_silence_` clips** are a random 1 s excerpt of a background-noise recording at a random volume in [0, 1). Training sets draw fresh ones every time. Evaluation sets use a fixed set seeded by the split name, so validation and test scores are reproducible. That fixed set is also identical in the 12-class and 13-class versions of a split.
- **Augmentation** applies to training only, and only to real clips (Google's recipe): a random shift of up to ±100 ms, zero-filled, and background noise on 80% of clips at a volume of up to 0.1.

**Check:** one batch, class balance over the epoch, that augmentation changes the audio, that silence is not empty, that evaluation batches are deterministic, and that the 12- and 13-class test sets share the same silence clips.

In [ ]:
class NoiseBank:
    """The long _background_noise_ recordings: source of _silence_ clips and of augmentation noise."""

    def __init__(self, recordings, clip_len=CLIP):
        self.recs = [torch.from_numpy(r.astype(np.float32) / 32768.0) for r in recordings.values() if len(r) > clip_len]
        self.L = clip_len

    def excerpts(self, n, gen=None):
        """n random 1 s excerpts: a recording chosen uniformly, then a uniform offset inside it."""
        if not self.recs or n == 0:
            return torch.zeros(n, self.L)
        which = torch.randint(0, len(self.recs), (n,), generator=gen).tolist()
        where = torch.rand(n, generator=gen).tolist()
        return torch.stack([self.recs[w][int(u * (len(self.recs[w]) - self.L)):][:self.L]
                            for w, u in zip(which, where)])

    def silence(self, n, gen=None):
        """_silence_ clips: an excerpt at a random volume in [0, 1)."""
        return self.excerpts(n, gen) * torch.rand(n, 1, generator=gen)


def augment(wav, noise, gen=None):
    """Training augmentation on a batch [B, L], on any device: random shift (zero-filled) + background noise."""
    cfg = CFG["augment"]
    B, L = wav.shape
    s = int(cfg["shift_ms"] * SR / 1000)
    k = torch.randint(-s, s + 1, (B,), generator=gen).to(wav.device)
    src = torch.arange(L, device=wav.device)[None, :] - k[:, None]            # out[t] = in[t - k]
    wav = torch.gather(wav, 1, src.clamp(0, L - 1)) * ((src >= 0) & (src < L))
    use = (torch.rand(B, generator=gen) < cfg["noise_prob"]).float()
    vol = (torch.rand(B, generator=gen) * cfg["noise_max_vol"] * use).to(wav.device)
    return (wav + vol[:, None] * noise.excerpts(B, gen).to(wav.device)).clamp(-1.0, 1.0)


class ClipSet:
    """One split in RAM. Item i < n_real is real clip i; item i >= n_real is _silence_ clip (i - n_real).
    Training sets draw new silence every time; evaluation sets use a fixed set seeded by the split name."""

    def __init__(self, wav16, labels, n_silence, noise, split, train):
        self.wav = torch.from_numpy(np.ascontiguousarray(wav16))
        self.n_real, self.n_silence = len(labels), int(n_silence)
        self.labels = torch.cat([torch.as_tensor(labels, dtype=torch.long),
                                 torch.full((self.n_silence,), SILENCE, dtype=torch.long)])
        self.noise, self.split, self.train = noise, split, train
        g = torch.Generator().manual_seed(zlib.crc32(split.encode()))
        self.fixed_silence = None if train else noise.silence(self.n_silence, g)

    def __len__(self):
        return self.n_real + self.n_silence

    def get(self, idx, device=DEVICE, aug=False, gen=None):
        """Audio [len(idx), CLIP] float in [-1, 1] on `device`."""
        idx = idx.cpu()
        out = torch.zeros(len(idx), CLIP, device=device)
        real = idx < self.n_real
        if real.any():
            w = self.wav[idx[real]].to(device).float() / 32768.0
            if aug:
                w = augment(w, self.noise, gen)
            out[real.to(device)] = w
        if (~real).any():
            j = idx[~real] - self.n_real
            sil = self.noise.silence(len(j), gen) if self.train else self.fixed_silence[j]
            out[(~real).to(device)] = sil.to(device)
        return out

    def batches(self, batch_size, device=DEVICE, shuffle=False, aug=False, seed=0, max_batches=None, drop_last=False):
        """Yield (audio [B, CLIP] on device, labels [B] on device)."""
        gen = torch.Generator().manual_seed(seed)
        order = torch.randperm(len(self), generator=gen) if shuffle else torch.arange(len(self))
        stop = len(self) - (len(self) % batch_size if drop_last and len(self) >= batch_size else 0)
        for b, i in enumerate(range(0, stop, batch_size)):
            if max_batches is not None and b >= max_batches:
                break
            idx = order[i:i + batch_size]
            yield self.get(idx, device, aug, gen), self.labels[idx].to(device)


NOISE_BANK = NoiseBank(NOISE)


def make_set(split, words=True, unknown=True, silence=True, new=False, drop_new_from_unknown=False, train=False):
    """A ClipSet from the cache. Labels: keywords 0..9, _unknown_ 10, _silence_ 11, new word 12."""
    sp, W = INDEX["splits"][split], WAVS[split]
    parts, labels = [], []
    if words:
        parts.append(W["words"])
        labels += list(sp["labels"])
    if unknown:
        keep = [i for i, r in enumerate(sp["unknown"]) if not (drop_new_from_unknown and r.startswith(NEW_WORD + "/"))]
        parts.append(W["unknown"][keep])
        labels += [UNKNOWN] * len(keep)
    if new:
        parts.append(W["new"])
        labels += [NEW] * len(sp["new"])
    wav16 = np.concatenate(parts) if parts else np.zeros((0, CLIP), np.int16)
    return ClipSet(wav16, labels, sp["n_silence"] if silence else 0, NOISE_BANK, split, train)


TRAIN12 = make_set("training", train=True)
VAL12 = make_set("validation")
TEST12 = make_set("testing")
AUG = CFG["augment"]["enabled"]
log(f"03 sets: train {len(TRAIN12)} | validation {len(VAL12)} | test {len(TEST12)} clips (12 classes)")

In [ ]:
# ---- CHECK 3: batches, balance, augmentation, silence, determinism
bs = CFG["train"]["batch_size"]
t0 = time.time()
wav, y = next(TRAIN12.batches(bs, shuffle=True, aug=AUG, seed=0))
print(f"one training batch: audio {tuple(wav.shape)} {wav.dtype} on {wav.device}, labels {tuple(y.shape)}, "
      f"{time.time() - t0:.2f} s")
check("03 batch shape and range", wav.shape == (min(bs, len(TRAIN12)), CLIP) and float(wav.abs().max()) <= 1.0)
check("03 labels within 0..11", int(y.min()) >= 0 and int(y.max()) <= 11)

hist = torch.bincount(TRAIN12.labels, minlength=12)
print("training clips per class:", dict(zip(CLASSES_12, hist.tolist())))
ratio = float(hist.max()) / max(float(hist.min()), 1.0)
check("03 classes roughly balanced (max/min < 1.6)", ratio < 1.6, f"max/min = {ratio:.2f}")

idx = torch.arange(min(8, TRAIN12.n_real))
clean = TRAIN12.get(idx, aug=False)
noisy = TRAIN12.get(idx, aug=True, gen=torch.Generator().manual_seed(1))
check("03 augmentation changes the audio", float((clean - noisy).abs().mean()) > 1e-4,
      f"mean |change| {float((clean - noisy).abs().mean()):.4f}")
sil = VAL12.get(torch.arange(VAL12.n_real, len(VAL12)))
check("03 _silence_ clips are not empty", VAL12.n_silence == 0 or float(sil.abs().mean()) > 0,
      f"{VAL12.n_silence} validation silence clips, mean |x| {float(sil.abs().mean()) if len(sil) else 0:.4f}")
a = next(VAL12.batches(16))[0]
b = next(VAL12.batches(16))[0]
check("03 evaluation batches are deterministic", torch.equal(a, b))
tmp13 = make_set("testing", new=True)
check("03 12- and 13-class test sets share the same silence clips",
      TEST12.n_silence == 0 or torch.equal(TEST12.fixed_silence, tmp13.fixed_silence))
del tmp13

if HAVE_PLT:
    fig, axes = plt.subplots(1, 2, figsize=(12, 2.2))
    axes[0].plot(clean[0].cpu(), lw=0.5)
    axes[0].set_title("clean")
    axes[1].plot(noisy[0].cpu(), lw=0.5)
    axes[1].set_title("augmented (shift + noise)")
    savefig("03_augmentation")

## 4 · MFCC front end, written from scratch

Audio [B, 16000] → 25 ms Hann window every 10 ms (101 frames) → power spectrum → 64 triangular mel filters (HTK scale, 20 Hz–8 kHz) → **clamp at the mel floor** → log → orthonormal DCT, keep 40 coefficients → z-score each coefficient.

These match torchaudio's defaults, which the original project used. If torchaudio is installed, the check cell compares the two.

- **The mel floor matters for the delta encoder.** Without it, the log of near-silence jitters from frame to frame, the delta encoder fires on that jitter, and the network stalls. It is set to the median mel power of training clips as the network sees them (augmented, as in the original project), never below 1e-3. The original checkpoint used 0.0055.
- **The normalisation statistics** are the per-coefficient mean and std over the same augmented training clips.

**Check:** filterbank shape with no empty filter, MFCC shape [B, 101, 40] and finite values, normalised features with mean ≈ 0 and std ≈ 1, and (if torchaudio is installed) agreement with torchaudio's MFCC.

Saved: `04_frontend/frontend.pt` (all buffers) and `frontend.json`.

In [ ]:
def hz_to_mel(f):
    return 2595.0 * np.log10(1.0 + np.asarray(f, dtype=np.float64) / 700.0)


def mel_to_hz(m):
    return 700.0 * (10.0 ** (np.asarray(m, dtype=np.float64) / 2595.0) - 1.0)


def mel_filterbank(n_fft, n_mels, sr, f_min, f_max):
    """Triangular filters on the HTK mel scale, shape [n_fft//2 + 1, n_mels] (torchaudio's default, norm=None)."""
    freqs = np.linspace(0, sr // 2, n_fft // 2 + 1)
    f_pts = mel_to_hz(np.linspace(hz_to_mel(f_min), hz_to_mel(f_max), n_mels + 2))   # edges and centres
    f_diff = np.diff(f_pts)
    slopes = f_pts[None, :] - freqs[:, None]
    down = -slopes[:, :-2] / f_diff[:-1]        # rising edge of each triangle
    up = slopes[:, 2:] / f_diff[1:]             # falling edge
    return torch.tensor(np.maximum(0.0, np.minimum(down, up)), dtype=torch.float32)


def dct_matrix(n_mfcc, n_mels):
    """Orthonormal DCT-II, shape [n_mels, n_mfcc]: log-mel energies -> cepstral coefficients."""
    n = np.arange(n_mels)
    k = np.arange(n_mfcc)[:, None]
    d = np.cos(np.pi / n_mels * (n + 0.5) * k)
    d[0] *= 1.0 / np.sqrt(2.0)
    d *= np.sqrt(2.0 / n_mels)
    return torch.tensor(d.T, dtype=torch.float32)


class MFCCFrontend(nn.Module):
    """audio [B, 16000] -> z-scored MFCC [B, 101, 40]. Every constant is a buffer, so .to(device) moves it."""

    def __init__(self, cfg):
        super().__init__()
        self.n_fft, self.hop = cfg["n_fft"], cfg["hop"]
        self.register_buffer("window", torch.hann_window(cfg["n_fft"]))
        self.register_buffer("fb", mel_filterbank(cfg["n_fft"], cfg["n_mels"], SR, cfg["f_min"], cfg["f_max"]))
        self.register_buffer("dct", dct_matrix(cfg["n_mfcc"], cfg["n_mels"]))
        self.register_buffer("mel_floor", torch.tensor(1e-5))
        self.register_buffer("mean", torch.zeros(cfg["n_mfcc"]))
        self.register_buffer("std", torch.ones(cfg["n_mfcc"]))

    def mel_power(self, wav):
        spec = torch.stft(wav, self.n_fft, hop_length=self.hop, win_length=self.n_fft, window=self.window,
                          center=True, pad_mode="reflect", return_complex=True)       # [B, 201, 101]
        power = spec.real ** 2 + spec.imag ** 2
        return power.transpose(1, 2) @ self.fb                                        # [B, 101, 64]

    def mfcc(self, wav):
        return torch.maximum(self.mel_power(wav), self.mel_floor).log() @ self.dct     # [B, 101, 40]

    def forward(self, wav):
        return (self.mfcc(wav) - self.mean) / self.std


@torch.no_grad()
def fit_frontend(fe, train_set):
    """Set the mel floor and the normalisation statistics from training clips."""
    cfg, bs = CFG["frontend"], CFG["train"]["batch_size"]
    vals = []
    for wav, _ in train_set.batches(bs, shuffle=True, aug=AUG, seed=1, max_batches=cfg["floor_batches"]):
        v = fe.mel_power(wav).flatten()
        vals.append(v[torch.randperm(v.numel(), device=v.device)[:200_000]].cpu())
    est = float(torch.cat(vals).median())
    floor = max(est, cfg["mel_floor_min"])
    fe.mel_floor.fill_(floor)
    s = torch.zeros(cfg["n_mfcc"], dtype=torch.float64)
    ss, n = torch.zeros_like(s), 0
    for wav, _ in train_set.batches(bs, shuffle=True, aug=AUG, seed=2, max_batches=cfg["stats_batches"]):
        m = fe.mfcc(wav).double().reshape(-1, cfg["n_mfcc"]).cpu()
        s += m.sum(0)
        ss += (m * m).sum(0)
        n += m.shape[0]
    mean = s / n
    std = (ss / n - mean ** 2).clamp_min(0).sqrt().clamp_min(1e-5)
    fe.mean.copy_(mean.float())
    fe.std.copy_(std.float())
    return {"mel_floor": floor, "median_mel_power": est}


d4 = stage_dir("04_frontend")
FE = MFCCFrontend(CFG["frontend"]).to(DEVICE)
if is_done("04_frontend"):
    FE.load_state_dict(load_pt(d4 / "frontend.pt"))
    log(f"04_frontend: loaded (mel floor {float(FE.mel_floor):.3g})")
else:
    info = fit_frontend(FE, TRAIN12)
    save_pt(FE.state_dict(), d4 / "frontend.pt")
    save_json({**info, "mean": FE.mean, "std": FE.std}, d4 / "frontend.json")
    mark_done("04_frontend", **info)
    log(f"04_frontend: mel floor {info['mel_floor']:.3g} (median mel power {info['median_mel_power']:.3g})")

In [ ]:
# ---- CHECK 4: filterbank, shapes, normalisation, and (optional) agreement with torchaudio
fb = FE.fb.cpu()
empty = int((fb.max(0).values == 0).sum())
check("04 mel filterbank shape and no empty filter",
      tuple(fb.shape) == (CFG["frontend"]["n_fft"] // 2 + 1, CFG["frontend"]["n_mels"]) and empty == 0,
      f"{tuple(fb.shape)}, {empty} empty filters")
wav, y = next(VAL12.batches(16))
with torch.no_grad():
    m = FE.mfcc(wav)
n_frames = 1 + CLIP // CFG["frontend"]["hop"]
check("04 MFCC shape [B, 101, 40] and finite",
      tuple(m.shape) == (len(wav), n_frames, CFG["frontend"]["n_mfcc"]) and bool(torch.isfinite(m).all()),
      str(tuple(m.shape)))
with torch.no_grad():
    zt = FE(next(TRAIN12.batches(64, shuffle=True, aug=AUG, seed=5))[0])
zm, zs = float(zt.mean()), float(zt.std())
check("04 normalised features: mean ~0, std ~1", abs(zm) < 0.3 and 0.6 < zs < 1.4, f"mean {zm:.3f}, std {zs:.3f}")
print("raw MFCC std per coefficient, c0 first:", [round(v, 2) for v in FE.std[:6].tolist()], "...")
try:
    import torchaudio
    fc = CFG["frontend"]
    ta_mel = torchaudio.transforms.MelSpectrogram(sample_rate=SR, n_fft=fc["n_fft"], hop_length=fc["hop"],
                                                  n_mels=fc["n_mels"], f_min=fc["f_min"], f_max=fc["f_max"]).to(DEVICE)
    ta_dct = torchaudio.functional.create_dct(fc["n_mfcc"], fc["n_mels"], "ortho").to(DEVICE)
    with torch.no_grad():
        ref = torch.maximum(ta_mel(wav), FE.mel_floor).log().transpose(1, 2) @ ta_dct
    diff = float((ref - m).abs().max())
    check("04 MFCC matches torchaudio (optional)", diff < 1e-2, f"max abs difference {diff:.2e}")
except ImportError:
    log("  torchaudio not installed: skipping the optional comparison (not needed)")
except Exception as e:                    # installed but broken or a different API version: still optional
    log(f"  torchaudio comparison skipped ({type(e).__name__}: {str(e)[:120]}); the notebook does not need torchaudio")

if HAVE_PLT:
    fig, axes = plt.subplots(1, 3, figsize=(13, 2.6))
    for ax, i in zip(axes, range(3)):
        ax.imshow(m[i].T.cpu(), aspect="auto", origin="lower")
        ax.set_title(f"MFCC, validation clip of class '{CLASSES_12[int(y[i])]}'")
        ax.set_xlabel("frame")
    savefig("04_mfcc")

## 5 · Delta encoder: 40 coefficients → 80 spike channels

Each coefficient keeps a reference level, which starts at its first frame. When the signal rises more than θ = 0.35 (z-units) above the reference, the **ON** channel spikes and the reference steps up by θ. The **OFF** channel does the same for falls. This is at most one spike per channel per frame, so a jump of height h produces ⌊h/θ⌋ spikes over consecutive frames. A constant signal never spikes, which is why silence costs almost nothing.

The network sees one 80-wide column per 10 ms step, for 101 steps.

**Check (small experiments):** unit tests on hand-made signals; the spike cost on near-silence vs speech (it must be ~0 on silence); how that cost changes with the mel floor (the trap that stalled the original project); and how it changes with θ. Saved in `05_encoder/encoder.json`.

In [ ]:
THETA = CFG["encoder"]["theta"]
N_IN = 2 * CFG["frontend"]["n_mfcc"]                 # 80 input channels
T_STEPS = 1 + CLIP // CFG["frontend"]["hop"]          # 101 time steps


def delta_encode(z, theta=THETA):
    """Level-crossing encoder. z: z-scored MFCC [B, T, F] -> spikes [T, B, 2F] (ON channels, then OFF)."""
    z = z.transpose(0, 1)                         # [T, B, F]
    ref = z[0].clone()
    out = torch.empty(z.shape[0], z.shape[1], 2 * z.shape[2], device=z.device)
    for t in range(z.shape[0]):
        d = z[t] - ref
        up, dn = (d > theta).float(), (d < -theta).float()
        ref = ref + theta * (up - dn)
        out[t] = torch.cat([up, dn], dim=-1)
    return out


@torch.no_grad()
def make_input(wav):
    """The whole front end: audio [B, 16000] -> 80 spike channels [101, B, 80]."""
    return delta_encode(FE(wav))


@torch.no_grad()
def spikes_per_frame(wav, theta=THETA):
    """Input spikes per 10 ms frame, summed over the 80 channels and averaged over frames and clips."""
    return float(delta_encode(FE(wav), theta).sum(-1).mean())

In [ ]:
# ---- CHECK 5: unit tests, silence cost, and two small experiments
d5 = stage_dir("05_encoder")
check("05 constant input -> no spikes", float(delta_encode(torch.zeros(1, 50, 3)).sum()) == 0.0)
h = 1.0
step = torch.zeros(1, 50, 1)
step[:, 10:] = h
sp = delta_encode(step)
n_on, n_off = int(sp[:, 0, 0].sum()), int(sp[:, 0, 1].sum())
check("05 a step of height h gives floor(h/theta) ON spikes and no OFF", n_on == int(math.floor(h / THETA)) and n_off == 0,
      f"{n_on} ON, {n_off} OFF for h={h}, theta={THETA}")
sp2 = delta_encode(-step)
check("05 a downward step gives OFF spikes", int(sp2[:, 0, 1].sum()) == int(math.floor(h / THETA)) and int(sp2[:, 0, 0].sum()) == 0)

quiet = (1e-4 * torch.randn(8, CLIP, generator=torch.Generator().manual_seed(0))).to(DEVICE)
speech = next(VAL12.batches(64))[0]
sil_cost, speech_cost = spikes_per_frame(quiet), spikes_per_frame(speech)
check("05 silence costs ~0 input spikes per frame (< 2)", sil_cost < 2.0,
      f"{sil_cost:.3f} on near-silence vs {speech_cost:.2f} on validation clips; if this fails, raise frontend.mel_floor_min")

# experiment A: the mel floor decides whether silence is free
orig = float(FE.mel_floor)
floor_rows = []
for factor in [1e-3, 1e-2, 1.0, 10.0]:
    FE.mel_floor.fill_(orig * factor)
    floor_rows.append({"floor": orig * factor, "silence_spf": spikes_per_frame(quiet), "speech_spf": spikes_per_frame(speech)})
FE.mel_floor.fill_(orig)
print(f"\n{'mel floor':>12} {'silence spikes/frame':>22} {'speech spikes/frame':>21}")
for r in floor_rows:
    print(f"{r['floor']:>12.2e} {r['silence_spf']:>22.3f} {r['speech_spf']:>21.2f}")

# experiment B: theta trades spikes for detail
theta_rows = [{"theta": th, "speech_spf": spikes_per_frame(speech, th), "silence_spf": spikes_per_frame(quiet, th)}
              for th in [0.2, 0.35, 0.5, 0.8]]
print(f"\n{'theta':>8} {'speech spikes/frame':>21} {'silence spikes/frame':>22}")
for r in theta_rows:
    print(f"{r['theta']:>8.2f} {r['speech_spf']:>21.2f} {r['silence_spf']:>22.3f}")

x_demo = make_input(speech[:1])[:, 0].cpu()
active = float((x_demo.sum(-1) > 0).float().mean())
save_json({"silence_spikes_per_frame": sil_cost, "speech_spikes_per_frame": speech_cost,
           "active_frame_fraction_example": active, "mel_floor_experiment": floor_rows,
           "theta_experiment": theta_rows}, d5 / "encoder.json")
if not is_done("05_encoder"):          # mark once: re-marking would make later stages look stale
    mark_done("05_encoder")
if HAVE_PLT:
    plt.figure(figsize=(12, 2.6))
    plt.imshow(x_demo.T, aspect="auto", origin="lower", cmap="Greys", interpolation="nearest")
    plt.title(f"delta spikes of one validation clip ('{CLASSES_12[int(VAL12.labels[0])]}'): rows 0-39 ON, 40-79 OFF")
    plt.xlabel("time step (10 ms)")
    savefig("05_delta_raster")

## 6 · The spiking network

80 inputs → three hidden layers of 512 leaky integrate-and-fire neurons → 12 leaky integrators.

- **Hidden neuron, one step:** U[t] = β·U[t−1]·(1 − S[t−1]) + W·x[t], and S[t] = 1 if U[t] > θ. Here β = 0.9 (a time constant of about 95 ms) and θ = 1. The reset uses the previous spike and carries no gradient, as in snnTorch's `Leaky`.
- **Readout:** V[t] = β·V[t−1] + W_out·S₃[t], with no reset. The logit for class c is the maximum of V_c over the 101 steps, and the loss is cross-entropy on those logits.
- **Backward pass:** the spike's derivative is replaced by the atan surrogate 1 / (1 + (π(U − θ))²) (snnTorch's `atan` with α = 2). There is no 1/π factor.
- **Neuron ids:** global ids 0…1535, printed as `L<layer>N<index>`, for example `L2N417`. A mask zeroes chosen neurons' spikes and membrane at every step; every ablation uses it.

**Check:** a single neuron fed a constant 0.5 must spike at steps 3, 6, 9, 12. The cell also checks the surrogate values, the parameter count (571,392), firing at initialisation, that gradients reach every layer, that masking layer 0 silences everything, and that a small "overfit two batches" run lowers the loss.

In [ ]:
class ATanSpike(torch.autograd.Function):
    """Forward: Heaviside step (spike if x > 0). Backward: 1 / (1 + (pi*x)^2), the derivative of
    arctan(pi*x)/pi + 1/2. It is largest near threshold, where a small change could flip the spike."""

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return (x > 0).to(x.dtype)

    @staticmethod
    def backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        return grad_out / (1.0 + (math.pi * x) ** 2)


spike_fn = ATanSpike.apply


def lif_step(mem, cur, beta, thr):
    """One step of a layer of LIF neurons with reset-to-zero:
        U[t] = beta * U[t-1] * (1 - S[t-1]) + I[t],   S[t] = 1 if U[t] > thr."""
    reset = (mem > thr).to(mem.dtype)            # spiked on the previous step? (no gradient)
    mem = beta * mem * (1.0 - reset) + cur
    return spike_fn(mem - thr), mem


class SpikingKWS(nn.Module):
    """80 spike inputs -> hidden LIF layers -> n_out leaky integrators (no reset).
    logit[c] = max over time of readout membrane c."""

    def __init__(self, n_in, hidden, n_out, beta, threshold):
        super().__init__()
        self.n_in, self.hidden, self.n_out = n_in, list(hidden), n_out
        self.beta, self.thr = beta, threshold
        sizes = [n_in] + self.hidden + [n_out]
        self.fc = nn.ModuleList(nn.Linear(a, b, bias=False) for a, b in zip(sizes[:-1], sizes[1:]))
        for lin in self.fc:
            nn.init.normal_(lin.weight, 0.0, math.sqrt(2.0 / lin.in_features))       # He init
        self.offsets = [int(v) for v in np.cumsum([0] + self.hidden)]                 # first global id of each layer
        self.n_hidden = self.offsets[-1]

    # ---- neuron addresses: global id <-> (layer, index within layer)
    def layer_of(self, gid):
        return int(np.searchsorted(self.offsets, gid, side="right") - 1)

    def name(self, gid):
        l = self.layer_of(gid)
        return f"L{l}N{gid - self.offsets[l]}"

    def masks_from_gids(self, gids, keep=False, always_on=(), device=DEVICE):
        """Per-layer 0/1 vectors. keep=False: silence `gids`. keep=True: silence all hidden neurons
        except `gids` and `always_on`."""
        m = torch.zeros(self.n_hidden) if keep else torch.ones(self.n_hidden)
        g = list(gids)
        if g:
            m[torch.as_tensor(g, dtype=torch.long)] = 1.0 if keep else 0.0
        a = list(always_on)
        if a:
            m[torch.as_tensor(a, dtype=torch.long)] = 1.0
        return [m[self.offsets[l]:self.offsets[l + 1]].to(device) for l in range(len(self.hidden))]

    def forward(self, x, masks=None, record=False):
        """x: input spikes [T, B, n_in]. Returns {"logits": [B, n_out], "counts": per-layer spike counts [B, N_l]}.
        record=True adds "spikes" (per layer, a list of T tensors [B, N_l] still in the autograd graph) and
        "mem_out" (readout membrane [T, B, n_out]); record="out" adds only "mem_out"."""
        T, B, _ = x.shape
        mems = [x.new_zeros(B, n) for n in self.hidden]
        counts = [x.new_zeros(B, n) for n in self.hidden]
        mem_out = x.new_zeros(B, self.n_out)
        rec_spk = [[] for _ in self.hidden] if record is True else None
        rec_out = []
        for t in range(T):
            h = x[t]
            for l in range(len(self.hidden)):
                s, mems[l] = lif_step(mems[l], self.fc[l](h), self.beta, self.thr)
                if masks is not None:
                    s, mems[l] = s * masks[l], mems[l] * masks[l]
                counts[l] = counts[l] + s.detach()
                if rec_spk is not None:
                    rec_spk[l].append(s)
                h = s
            mem_out = self.beta * mem_out + self.fc[-1](h)          # leaky integrator, never reset
            rec_out.append(mem_out)
        M = torch.stack(rec_out)                                      # [T, B, n_out]
        out = {"logits": M.max(0).values, "counts": counts}          # max over time, as snnTorch's ce_max_membrane_loss
        if rec_spk is not None:
            out["spikes"] = rec_spk
        if record:
            out["mem_out"] = M
        return out


def new_model(n_out=12):
    return SpikingKWS(N_IN, CFG["net"]["hidden"], n_out, CFG["net"]["beta"], CFG["net"]["threshold"]).to(DEVICE)


def expand_readout(model, new_row):
    """A copy of `model` with one more output (label model.n_out). Old weights are copied unchanged;
    the new readout row starts at `new_row`."""
    m = new_model(model.n_out + 1)
    with torch.no_grad():
        for a, b in zip(model.fc[:-1], m.fc[:-1]):
            b.weight.copy_(a.weight)
        m.fc[-1].weight[:model.n_out] = model.fc[-1].weight
        m.fc[-1].weight[model.n_out] = new_row.to(m.fc[-1].weight.device)
    return m

In [ ]:
# ---- CHECK 6: unit tests of the neuron and the network
d6 = stage_dir("06_model")
mem, spk = torch.zeros(1), []
for t in range(12):
    s, mem = lif_step(mem, torch.tensor([0.5]), 0.9, 1.0)
    spk.append(int(s))
steps = [i + 1 for i, s in enumerate(spk) if s]
check("06 LIF: constant 0.5 spikes at steps 3, 6, 9, 12 (U = 0.50, 0.95, 1.355, reset)", steps == [3, 6, 9, 12], str(steps))

xg = torch.tensor([-1.0, -0.1, 0.0, 0.1, 1.0], requires_grad=True)
spike_fn(xg).sum().backward()
expect = 1.0 / (1.0 + (math.pi * xg.detach()) ** 2)
check("06 surrogate gradient = 1/(1+(pi x)^2)", torch.allclose(xg.grad, expect),
      str([round(v, 3) for v in xg.grad.tolist()]))

seed_all(CFG["seed"])
probe = new_model()
n_par = sum(p.numel() for p in probe.parameters())
expected = sum(a * b for a, b in zip([N_IN] + CFG["net"]["hidden"], CFG["net"]["hidden"] + [12]))
check("06 parameter count", n_par == expected, f"{n_par:,} weights (571,392 for 80-512-512-512-12)")

wav, y = next(VAL12.batches(32))
x = make_input(wav)
with torch.no_grad():
    out = probe(x)
rates = [float(c.mean()) / T_STEPS for c in out["counts"]]
check("06 every hidden layer fires at initialisation", all(r > 1e-4 for r in rates),
      f"spike probability per neuron and step: {[round(r, 4) for r in rates]}")
check("06 logits are finite", bool(torch.isfinite(out["logits"]).all()))

out = probe(x, record=True)
loss = F.cross_entropy(out["logits"], y)
g_mid = torch.autograd.grad(loss, [out["spikes"][l][T_STEPS // 2] for l in range(len(probe.hidden))], retain_graph=True)
loss.backward()
check("06 gradient reaches every weight matrix", all(float(lin.weight.grad.abs().sum()) > 0 for lin in probe.fc),
      "dL/ds at the middle step, per layer: " + str([f"{float(g.abs().mean()):.2e}" for g in g_mid]))

m0 = probe.masks_from_gids(range(probe.offsets[0], probe.offsets[1]))      # silence all of layer 0
with torch.no_grad():
    o = probe(x, masks=m0)
check("06 masking layer 0 silences the whole network",
      all(float(c.sum()) == 0 for c in o["counts"]) and float(o["logits"].abs().max()) == 0)

try:                                              # optional: same weights through snnTorch's Leaky neurons
    import snntorch as snn
    from snntorch import surrogate as snn_surrogate
    kw = dict(beta=CFG["net"]["beta"], threshold=CFG["net"]["threshold"], spike_grad=snn_surrogate.atan(), init_hidden=False)
    leaky = [snn.Leaky(reset_mechanism="zero", **kw) for _ in probe.hidden] + [snn.Leaky(reset_mechanism="none", **kw)]
    leaky = [m.to(DEVICE) for m in leaky]

    def snntorch_logits(xx, net):
        mems = [xx.new_zeros(xx.shape[1], lin.out_features) for lin in net.fc]
        outs = []
        for t in range(xx.shape[0]):
            h = xx[t]
            for l, lin in enumerate(net.fc):
                h, mems[l] = leaky[l](lin(h), mems[l])
            outs.append(mems[-1])
        return torch.stack(outs).max(0).values

    mine, ref = probe(x)["logits"], snntorch_logits(x, probe)
    g_m = torch.autograd.grad(F.cross_entropy(mine, y), [lin.weight for lin in probe.fc])
    g_r = torch.autograd.grad(F.cross_entropy(ref, y), [lin.weight for lin in probe.fc])
    fdiff = float((mine - ref).abs().max())
    gdiff = max(float((a - b).norm() / b.norm().clamp_min(1e-12)) for a, b in zip(g_m, g_r))
    check("06 network matches snnTorch Leaky, forward and gradients (optional)", fdiff < 1e-4 and gdiff < 1e-3,
          f"logit max difference {fdiff:.1e}, weight-gradient relative difference {gdiff:.1e}")
except ImportError:
    log("  snnTorch not installed: skipping the optional comparison (not needed)")
except Exception as e:                    # installed but incompatible: still optional
    log(f"  snnTorch comparison skipped ({type(e).__name__}: {str(e)[:120]}); the notebook does not need snnTorch")

tiny = new_model()
opt = torch.optim.Adam(tiny.parameters(), lr=4 * CFG["train"]["lr"])
fixed = [(make_input(w), yy) for w, yy in TRAIN12.batches(32, shuffle=True, seed=3, max_batches=2)]
losses = []
for it in range(30):
    xx, yy = fixed[it % len(fixed)]
    l = F.cross_entropy(tiny(xx)["logits"], yy)
    opt.zero_grad()
    l.backward()
    opt.step()
    losses.append(l.item())
check("06 tiny overfit: loss falls on 2 fixed batches", np.mean(losses[-4:]) < 0.8 * np.mean(losses[:4]),
      f"{np.mean(losses[:4]):.3f} -> {np.mean(losses[-4:]):.3f}")
save_json({"n_params": n_par, "hidden": CFG["net"]["hidden"], "init_rates": rates, "overfit_losses": losses},
          d6 / "model_card.json")
if not is_done("06_model"):
    mark_done("06_model")
del probe, tiny, opt, fixed

## 7 · Base training (BPTT) with early stopping

Adam at lr = 5e-4 · 256 / 512 = 2.5e-4, cosine decay over a **100-epoch cap**, batch 128, augmentation on. After every epoch the network is scored on **validation**, now including the validation **loss** (cross-entropy of the max-membrane logits). Training **stops once the validation loss has not improved for 3 epochs in a row**, and the network with the lowest validation loss is kept as `best.pt`. `last.pt` is written every epoch with the optimiser state and the early-stopping counter, so an interrupted run resumes where it stopped.

**Check:** as in the pipeline notebook, plus whether training ended by early stopping or by the cap.

In [ ]:
def decide(logits):
    """The predicted class, or -1 ("no decision") when several classes share the top logit.
    A clip whose readout never gets a spike has every logit exactly 0; a plain argmax would call it class 0
    ("yes") every time, which inflates "yes" whenever an ablation silences the network."""
    top = logits.max(-1, keepdim=True).values
    tied = (logits == top).sum(-1) > 1
    return torch.where(tied, torch.full_like(logits.argmax(-1), -1), logits.argmax(-1))


@torch.no_grad()
def evaluate(model, data, n_classes=12, masks=None, batch_size=None):
    """Accuracy, per-class accuracy, confusion matrix [true, predicted] and mean cross-entropy on a ClipSet.
    "No decision" clips (tied top logits) count as wrong and are left out of the confusion matrix;
    acc_argmax is the plain-argmax accuracy (ties -> class 0), comparable with the original project."""
    model.eval()
    conf = torch.zeros(n_classes, n_classes, dtype=torch.long)
    total = torch.zeros(n_classes, dtype=torch.long)
    tied = torch.zeros(n_classes, dtype=torch.long)
    hit_argmax, loss_sum, n_loss = 0, 0.0, 0
    for wav, y in data.batches(batch_size or CFG["train"]["eval_batch_size"]):
        logits = model(make_input(wav), masks=masks)["logits"]
        yd = y.to(logits.device)
        known = yd < logits.shape[1]                      # a 12-output net cannot be scored on label 12
        if bool(known.any()):
            loss_sum += float(F.cross_entropy(logits[known].float(), yd[known], reduction="sum"))
            n_loss += int(known.sum())
        pred, y = decide(logits).cpu().clamp(max=n_classes - 1), y.cpu()
        ok = pred >= 0
        conf += torch.bincount(y[ok] * n_classes + pred[ok], minlength=n_classes * n_classes).view(n_classes, n_classes)
        total += torch.bincount(y, minlength=n_classes)
        tied += torch.bincount(y[~ok], minlength=n_classes)
        hit_argmax += int((logits.argmax(-1).cpu() == y).sum())
    n = max(int(total.sum()), 1)
    per_class = (conf.diagonal().double() / total.clamp_min(1).double()).tolist()
    return {"acc": float(conf.diagonal().sum()) / n, "per_class": per_class, "confusion": conf.tolist(),
            "n": int(total.sum()), "n_tied": int(tied.sum()), "tied_per_class": tied.tolist(), "acc_argmax": hit_argmax / n,
            "loss": loss_sum / max(n_loss, 1)}


def train_one_epoch(model, opt, epoch):
    model.train()
    cfgt = CFG["train"]
    hit = tot = nb = 0
    loss_sum, rates, t0 = 0.0, np.zeros(len(model.hidden)), time.time()
    for wav, y in TRAIN12.batches(cfgt["batch_size"], shuffle=True, aug=AUG, seed=CFG["seed"] * 100003 + epoch,
                                  max_batches=cfgt["max_train_batches"], drop_last=True):
        out = model(make_input(wav))
        loss = F.cross_entropy(out["logits"], y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        hit += int((decide(out["logits"].detach()) == y).sum())
        tot += len(y)
        nb += 1
        loss_sum += loss.item() * len(y)
        rates += np.array([float(c.mean()) / T_STEPS for c in out["counts"]])
    sec = time.time() - t0
    return {"loss": loss_sum / max(tot, 1), "train_acc": hit / max(tot, 1),
            "rates": (rates / max(nb, 1)).round(4).tolist(), "sec": round(sec, 1),
            "clips_per_s": round(tot / max(sec, 1e-9), 1)}


d7 = stage_dir("07_train")


def train_base():
    """Adam with a cosine schedule over the epoch cap. Stops once the validation LOSS has not improved for
    `early_stop_patience` epochs in a row; the lowest-validation-loss network is kept as best.pt."""
    cfgt = CFG["train"]
    patience = cfgt["early_stop_patience"]
    seed_all(CFG["seed"])
    model = new_model()
    opt = torch.optim.Adam(model.parameters(), lr=cfgt["lr"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfgt["epochs"])
    start, history, best_loss, bad = 0, [], float("inf"), 0
    basis = stage_basis("07_train")
    if (d7 / "last.pt").exists() and "07_train" not in FORCE:
        ck = load_pt(d7 / "last.pt")
        if ck.get("basis") == basis:
            model.load_state_dict(ck["model"])
            opt.load_state_dict(ck["opt"])
            sched.load_state_dict(ck["sched"])
            start, history, best_loss, bad = ck["epoch"] + 1, ck["history"], ck["best_loss"], ck["bad"]
            log(f"07_train: resuming after epoch {start} ({bad} epoch(s) without improvement so far)")
        else:
            log("07_train: last.pt was trained with other settings or inputs; starting over")
    for epoch in range(start, cfgt["epochs"]):
        if bad >= patience:
            break
        row = {"epoch": epoch + 1, "lr": opt.param_groups[0]["lr"], **train_one_epoch(model, opt, epoch)}
        ev = evaluate(model, VAL12)
        row.update(val_acc=ev["acc"], val_loss=ev["loss"])
        if ev["loss"] < best_loss:
            best_loss, bad, row["best"] = ev["loss"], 0, True
            save_pt({"model": model.state_dict(), "epoch": epoch + 1, "val_acc": ev["acc"], "val_loss": ev["loss"],
                     "cfg": CFG}, d7 / "best.pt")
        else:
            bad += 1
        sched.step()
        history.append(row)
        save_json(history, d7 / "history.json")
        save_pt({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "epoch": epoch,
                 "history": history, "best_loss": best_loss, "bad": bad, "basis": basis}, d7 / "last.pt")
        log(f"  ep {epoch + 1:3d}/{cfgt['epochs']} | loss {row['loss']:.3f} | train {row['train_acc']:.3f} | val loss "
            f"{ev['loss']:.4f} acc {ev['acc']:.3f} (best loss {best_loss:.4f}; {bad} without improvement) | "
            f"rates {row['rates']} | {row['sec']:.0f} s, {row['clips_per_s']:.0f} clips/s")
        if len(history) >= 2 and all(abs(r["loss"] - math.log(12)) < 0.01 for r in history[-2:]):
            check("07 training did not stall at ln(12)", False, "loss stuck at 2.485: every logit is equal. Lower the learning rate.")
            raise RuntimeError("training stalled at ln(12): every logit is equal. Lower CFG['train']['lr'].")
        if min(row["rates"]) < 1e-5:
            log("  warning: a hidden layer is (almost) silent; no gradient flows through a silent layer")
        if bad >= patience:
            log(f"  early stop after epoch {epoch + 1}: validation loss has not improved for {patience} epochs "
                f"(best {best_loss:.4f})")
    check("07 training did not stall at ln(12)", True, f"{len(history)} epochs, final loss {history[-1]['loss']:.3f}")
    return history


if is_done("07_train"):
    log("07_train: loaded best.pt")
else:
    train_base()
ck = load_pt(d7 / "best.pt")
MODEL = new_model()
MODEL.load_state_dict(ck["model"])
MODEL.eval()                     # the trained base network; later stages only read it or copy it
if is_done("07_train"):
    BASE_EVAL = load_json(d7 / "base_eval.json")
else:
    BASE_EVAL = {"best_epoch": ck["epoch"], "validation": evaluate(MODEL, VAL12), "testing": evaluate(MODEL, TEST12)}
    save_json(BASE_EVAL, d7 / "base_eval.json")
    mark_done("07_train", best_epoch=ck["epoch"], val_acc=BASE_EVAL["validation"]["acc"], test_acc=BASE_EVAL["testing"]["acc"])
log(f"07_train: best epoch {BASE_EVAL['best_epoch']} | validation {BASE_EVAL['validation']['acc']:.4f} "
    f"(selected on it) | test {BASE_EVAL['testing']['acc']:.4f} (held out)")

In [ ]:
# ---- CHECK 7: learning curves, per-class accuracy, confusions, and the readout rule
H = load_json(d7 / "history.json")
val, test = BASE_EVAL["validation"], BASE_EVAL["testing"]
check("07 validation accuracy well above chance (> 3/12)", val["acc"] > 3 / 12,
      f"validation {val['acc']:.4f}, test {test['acc']:.4f}, best epoch {BASE_EVAL['best_epoch']}")
check("07 training loss went down", H[-1]["loss"] < H[0]["loss"], f"{H[0]['loss']:.3f} -> {H[-1]['loss']:.3f}")
print(f"\n{'class':<12}{'validation':>12}{'test':>10}")
for c, nm in enumerate(CLASSES_12):
    print(f"{nm:<12}{val['per_class'][c]:>12.3f}{test['per_class'][c]:>10.3f}")
conf = np.array(test["confusion"])
off = [(int(conf[i, j]), CLASSES_12[i], CLASSES_12[j]) for i in range(12) for j in range(12) if i != j]
print("\nlargest test confusions (count, true -> predicted):", sorted(off, reverse=True)[:6])
print(f"clips with no decision (every logit tied, usually a silent readout): validation {val.get('n_tied', 0)}, "
      f"test {test.get('n_tied', 0)} {dict((c, n) for c, n in zip(CLASSES_12, test.get('tied_per_class', [])) if n)}")
print(f"plain-argmax accuracy (ties -> 'yes', as in the original project): validation {val.get('acc_argmax', float('nan')):.4f}, "
      f"test {test.get('acc_argmax', float('nan')):.4f}")


@torch.no_grad()
def readout_rule_accuracy(model, data):
    """Same weights, three ways to read the readout membrane. Only 'max' matches training."""
    hits, n = {"max (trained)": 0, "mean": 0, "last": 0}, 0
    for wav, y in data.batches(CFG["train"]["eval_batch_size"]):
        M = model(make_input(wav), record="out")["mem_out"]
        hits["max (trained)"] += int((decide(M.max(0).values) == y).sum())
        hits["mean"] += int((decide(M.mean(0)) == y).sum())
        hits["last"] += int((decide(M[-1]) == y).sum())
        n += len(y)
    return {k: v / max(n, 1) for k, v in hits.items()}


rules = readout_rule_accuracy(MODEL, VAL12)
print("\nvalidation accuracy by readout rule:", {k: round(v, 4) for k, v in rules.items()})
check("07 max-membrane readout reproduces the stored validation accuracy",
      abs(rules["max (trained)"] - val["acc"]) < 0.002, f"{rules['max (trained)']:.4f} vs {val['acc']:.4f}")
BASE_EVAL["readout_rules_validation"] = rules
save_json(BASE_EVAL, d7 / "base_eval.json")

if HAVE_PLT:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3))
    ep = [r["epoch"] for r in H]
    axes[0].plot(ep, [r["loss"] for r in H])
    axes[0].axhline(math.log(12), ls="--", c="grey", lw=0.8)
    axes[0].set_title("training loss (dashed: ln 12 = stalled)")
    axes[1].plot(ep, [r["train_acc"] for r in H], label="train")
    axes[1].plot([r["epoch"] for r in H if "val_acc" in r], [r["val_acc"] for r in H if "val_acc" in r], label="validation")
    axes[1].legend()
    axes[1].set_title("accuracy")
    axes[2].plot(ep, [r["rates"] for r in H])
    axes[2].set_title("hidden spike probability per step")
    axes[2].legend([f"layer {l}" for l in range(len(MODEL.hidden))])
    savefig("07_training")
    plt.figure(figsize=(5.5, 4.5))
    cn = conf / np.maximum(conf.sum(1, keepdims=True), 1)
    plt.imshow(cn, cmap="Blues", vmin=0, vmax=1)
    plt.xticks(range(12), CLASSES_12, rotation=90)
    plt.yticks(range(12), CLASSES_12)
    plt.title("test confusion (row = true class)")
    savefig("07_confusion")
stopped_early = len(H) < CFG["train"]["epochs"]
best_ep = min(H, key=lambda r: r.get("val_loss", float("inf")))["epoch"]
print(f"\ntraining ran {len(H)} epochs ({'early stop' if stopped_early else 'reached the cap'}); lowest validation loss "
      f"{min(r['val_loss'] for r in H):.4f} at epoch {best_ep}")
check("07 training ended by early stopping, not the epoch cap (a finding, not a bug, if it fails)", stopped_early,
      f"{len(H)} of {CFG['train']['epochs']} epochs; best validation loss at epoch {best_ep}")


## 8 · Per-class Fisher, Omega and the unique fraction (validation split)

For every hidden neuron i and class c:

- **Naive Fisher (EWC's importance):** Fᶜᵢᵢ = E[gᵢ²], with gᵢ = ∂L/∂sᵢ(t). Here sᵢ is the neuron's spike output at step t, and L is the cross-entropy of that clip with its true label (the empirical Fisher). The expectation runs over class-c clips and all 101 steps. The loss is summed, not averaged, over the batch, so each clip contributes its own gradient.
- **Omega (redundancy-corrected):** Ωᶜᵢ = 1 / (Fᶜ + λI)⁻¹ᵢᵢ = λ + Fᶜᵢᵢ·(1 − Rᵢ²). This is the loss sensitivity left after every other neuron re-adjusts. It is computed exactly from the full 1,536 × 1,536 Fisher matrix, with ridge λ = 1e-2 × the mean diagonal. λ is the same for every neuron of a class, so it does not change the Omega ranking.
- **Unique fraction:** 1 − Rᵢ² = (Ω − λ) / F. This is the part of a neuron's effect that no other neuron duplicates. The original project used Ω / F, which still contains λ: for a neuron with small Fᵢᵢ it is mostly λ / Fᵢᵢ, often clamped to 1. On the original run's Fisher matrices, 1,141 (class, neuron) pairs had Ω / F > 0.5, and none do once λ is removed.

Saved: `08_fisher/fisher.pt` holds naive, Omega and unique as [12, 1536] tensors, plus, if `save_full_fisher`, the full matrices (113 MB). `fisher_summary.json` holds the key numbers.

**Check:** per-layer share of the Fisher mass (the original project found 77% in layer 0), which layers each method's top 40 come from, condition numbers, and a ridge sweep showing how stable the rankings are.

In [ ]:
def fisher_per_class(model, data, n_classes=12):
    """Fisher[c] = mean over class-c clips and time steps of g g^T, g = dL/ds(t) over all hidden neurons."""
    bs = max(8, CFG["train"]["batch_size"] // 2)
    N, L = model.n_hidden, len(model.hidden)
    fsum = torch.zeros(n_classes, N, N, device=DEVICE)
    vecs, clips = torch.zeros(n_classes), torch.zeros(n_classes)
    t0 = time.time()
    for b, (wav, y) in enumerate(data.batches(bs)):
        x = make_input(wav)
        out = model(x, record=True)
        loss = F.cross_entropy(out["logits"], y, reduction="sum")          # sum: each clip's own gradient
        flat = [s for layer in out["spikes"] for s in layer]                  # L*T tensors [B, N_l]
        grads = torch.autograd.grad(loss, flat, allow_unused=True)
        grads = [torch.zeros_like(s) if g is None else g for s, g in zip(flat, grads)]
        T = x.shape[0]
        G = torch.cat([torch.stack(grads[l * T:(l + 1) * T], dim=1) for l in range(L)], dim=-1)   # [B, T, N]
        for c in y.unique().tolist():
            Gc = G[y == c].reshape(-1, N)
            fsum[c] += Gc.T @ Gc
            vecs[c] += Gc.shape[0]
            clips[c] += int((y == c).sum())
        if (b + 1) % 10 == 0:
            log(f"    fisher: {b + 1} batches, {time.time() - t0:.0f} s")
    return fsum.cpu().double() / vecs.clamp_min(1).double()[:, None, None], vecs, clips


def omega_from_fisher(Fc, ridge, cond=True):
    """Omega_i = 1 / ((F + lam I)^-1)_ii = lam + F_ii (1 - R_i^2), lam = ridge * mean(diag F).
    Returns (omega, naive = diag F, lam, condition number or None)."""
    d = Fc.diagonal().clone()
    lam = ridge * float(d.mean().clamp_min(1e-30))
    Fr = Fc + lam * torch.eye(Fc.shape[0], dtype=Fc.dtype)
    try:
        Finv = torch.cholesky_inverse(torch.linalg.cholesky(Fr))
    except RuntimeError:
        Finv = torch.linalg.pinv(Fr)
    omega = 1.0 / Finv.diagonal().clamp_min(1e-30)
    if not cond:
        return omega, d, lam, None
    ev = torch.linalg.eigvalsh(Fr)
    return omega, d, lam, float(ev.max() / ev.min().clamp_min(1e-30))


def ridge_sweep(FM, omega, ridge):
    """How much each class's top-K Omega set changes when the ridge is 100x smaller ... 10x larger."""
    base_top = [set(torch.argsort(omega[c], descending=True)[:K].tolist()) for c in range(len(FM))]
    rows = []
    for r in [ridge * 0.01, ridge * 0.1, ridge * 0.3, ridge * 3, ridge * 10]:
        ov, l0 = [], 0
        for c in range(len(FM)):
            o = omega_from_fisher(FM[c], r, cond=False)[0]
            top = torch.argsort(o, descending=True)[:K]
            ov.append(len(set(top.tolist()) & base_top[c]) / K)
            l0 += int((top < MODEL.offsets[1]).sum())
        rows.append({"ridge": r, "mean_overlap": float(np.mean(ov)), "min_overlap": float(np.min(ov)), "layer0_picks": l0})
    return rows


K = CFG["analysis"]["topk"]                       # neurons per class group
N_H, L_H = MODEL.n_hidden, len(MODEL.hidden)        # 1536 hidden neurons in 3 layers
layer_sl = [slice(MODEL.offsets[l], MODEL.offsets[l + 1]) for l in range(L_H)]

d8 = stage_dir("08_fisher")
if is_done("08_fisher"):
    FISH = load_pt(d8 / "fisher.pt")
    log("08_fisher: loaded")
else:
    t0 = time.time()
    FMATS, vecs, clips = fisher_per_class(MODEL, VAL12)
    om, nv, lams, cond = [], [], [], []
    for c in range(12):
        o, d, lam, cd = omega_from_fisher(FMATS[c], CFG["analysis"]["ridge"])
        om.append(o)
        nv.append(d)
        lams.append(lam)
        cond.append(cd)
    om, nv, lam_c = torch.stack(om), torch.stack(nv), torch.tensor(lams, dtype=torch.float64)[:, None]
    FISH = {"naive": nv.float(), "omega": om.float(),
            "unique": ((om - lam_c) / nv.clamp_min(1e-30)).clamp(0, 1).float(),     # 1 - R^2, ridge removed
            "lam": lams, "cond": cond, "vectors": vecs, "clips": clips, "ridge": CFG["analysis"]["ridge"],
            "ridge_sweep": ridge_sweep(FMATS, om, CFG["analysis"]["ridge"])}
    if CFG["analysis"]["save_full_fisher"]:
        FISH["F"] = FMATS.float()
    save_pt(FISH, d8 / "fisher.pt")
    mark_done("08_fisher", seconds=round(time.time() - t0))
    log(f"08_fisher: done in {time.time() - t0:.0f} s")

In [ ]:
# ---- CHECK 8: where Fisher puts importance, and how stable the rankings are
naive, omega, uniq = FISH["naive"].double(), FISH["omega"].double(), FISH["unique"].double()
check("08 Fisher and Omega are finite and non-negative",
      bool(torch.isfinite(naive).all() and torch.isfinite(omega).all() and (naive >= 0).all() and (omega >= 0).all()))
ratio = float(FISH["vectors"].min()) / N_H
check("08 enough gradient vectors per class (> 4 per dimension)", ratio > 4,
      f"worst class: {int(FISH['vectors'].min())} vectors for {N_H} neurons ({ratio:.1f}:1)", real_data_only=True)
share = torch.stack([naive[:, s].sum(1) / naive.sum(1) for s in layer_sl], 1)           # [12, L]
mean_f = [float(naive[:, s].mean()) for s in layer_sl]
print("share of naive Fisher mass per layer (mean over classes):", [round(v, 3) for v in share.mean(0).tolist()])
print("mean F_ii per layer:", [f"{v:.2e}" for v in mean_f],
      f"-> layer 0 / last layer = {mean_f[0] / max(mean_f[-1], 1e-30):.1f}x")


def picks_per_layer(scores, k=K):
    top = torch.argsort(scores, dim=1, descending=True)[:, :k]
    return [int(((top >= MODEL.offsets[l]) & (top < MODEL.offsets[l + 1])).sum()) for l in range(L_H)]


print(f"top-{K} picks per layer over 12 classes: naive {picks_per_layer(naive)}, Omega {picks_per_layer(omega)}")
print(f"condition numbers after the ridge: {min(FISH['cond']):.2e} .. {max(FISH['cond']):.2e}")
med_u = uniq.median(dim=1).values
n_u = (uniq > 0.5).sum(1)
lam_c = torch.tensor(FISH["lam"], dtype=torch.float64)[:, None]
ridge_share = float((lam_c / omega.clamp_min(1e-30)).median())
print("unique fraction (1 - R^2, ridge removed) median per class:", [round(v, 3) for v in med_u.tolist()])
print("neurons with more than half their effect unique, per class:", n_u.tolist())
print(f"share of Omega that is the ridge itself (median): {ridge_share:.0%}  "
      f"(Omega / F, as in the original project, would count it as unique)")
check("08 unique fraction is a proper fraction in [0, 1]", bool(((uniq >= 0) & (uniq <= 1)).all()))

ridge_rows = FISH.get("ridge_sweep", [])
print(f"\nridge sweep: top-{K} Omega overlap with the ridge actually used ({FISH['ridge']:g})")
for r in ridge_rows:
    print(f"  ridge {r['ridge']:8.1e}: mean overlap {r['mean_overlap']:5.0%}, worst class {r['min_overlap']:5.0%}, "
          f"layer-0 picks {r['layer0_picks']}/{12 * K}")
print("  The layer conclusion is trustworthy if layer-0 picks stay put; exact neurons move more.")
save_json({"layer_share": share.mean(0), "mean_F_per_layer": mean_f, "picks_per_layer_naive": picks_per_layer(naive),
           "picks_per_layer_omega": picks_per_layer(omega), "cond": FISH["cond"], "unique_median": med_u,
           "n_unique_gt_half": n_u, "ridge_share_of_omega": ridge_share, "ridge_sweep": ridge_rows}, d8 / "fisher_summary.json")

## 9 · Per-class mutual information (validation split)

This measure uses no gradients and no loss, only spike counts and labels.

1. Count each hidden neuron's spikes in every validation clip.
2. Put each neuron's counts into 8 **quantile** bins. Counts are skewed, so equal-width bins would put almost every clip in the first bin.
3. **Global MI:** I(a; Y) against the 12-class label, in nats; at most H(Y) = ln 12 = 2.485.
4. **One-vs-rest MI per class:** I(a; 1[Y = c]). Its top 40 neurons are class c's group.
5. **Shuffle null:** the labels are shuffled 20 times. A neuron counts as informative only if it beats the largest shuffled value.

Saved: `09_mi/mi.pt` holds global MI [1536], per-class MI [12, 1536], the null, and the spike counts as uint8.

**Check:** unit tests of the MI estimator on synthetic data with a known answer. Also the dead-neuron count, the null, mean MI per layer, the estimator bias for one-vs-rest, the sensitivity to the bin count, and the rank correlation with Fisher, overall and within each layer. Within-layer correlation matters because the overall correlation mostly reflects depth.

In [ ]:
@torch.no_grad()
def harvest_counts(model, data):
    """Spike count of every hidden neuron in every clip: [n_clips, N_hidden], plus the labels."""
    A, Y = [], []
    for wav, y in data.batches(CFG["train"]["eval_batch_size"]):
        A.append(torch.cat(model(make_input(wav))["counts"], dim=1).cpu())
        Y.append(y.cpu())
    return torch.cat(A), torch.cat(Y)


def quantile_bins(act, bins):
    """Per-neuron quantile binning [n, N] -> integers 0..bins-1. Constant neurons land in one bin
    and correctly get zero information."""
    qs = torch.linspace(0, 1, bins + 1)[1:-1]
    out = torch.zeros(act.shape, dtype=torch.long)
    for j in range(0, act.shape[1], 256):                 # chunks keep torch.quantile under its size limit
        a = act[:, j:j + 256].float()
        edges = torch.quantile(a, qs, dim=0)             # [bins-1, chunk]
        out[:, j:j + 256] = (a[None] > edges[:, None, :]).sum(0)
    return out


def mutual_information(disc, y, bins, n_classes):
    """I(a_i; Y) in nats for every neuron at once, from the (bin x label) contingency table."""
    n, N = disc.shape
    codes = (disc * n_classes + y[:, None]).T.contiguous()             # [N, n]
    hist = torch.zeros(N, bins * n_classes, dtype=torch.float64)
    hist.scatter_add_(1, codes, torch.ones(N, n, dtype=torch.float64))
    p = (hist / n).view(N, bins, n_classes)
    pa, py = p.sum(2, keepdim=True), p.sum(1, keepdim=True)
    return (p * (p.clamp_min(1e-30) / (pa * py).clamp_min(1e-30)).log()).sum((1, 2))


def avg_ranks(x):
    """Ranks 0..n-1 with ties sharing their average rank (MI has many exact zeros)."""
    _, inv, counts = torch.unique(torch.as_tensor(x, dtype=torch.float64).flatten(), return_inverse=True, return_counts=True)
    ends = torch.cumsum(counts, 0).double()
    return ((2.0 * ends - counts.double() - 1.0) / 2.0)[inv]


def spearman(u, v):
    ru, rv = avg_ranks(u), avg_ranks(v)
    ru, rv = ru - ru.mean(), rv - rv.mean()
    return float((ru @ rv) / (ru.norm() * rv.norm()).clamp_min(1e-30))


d9 = stage_dir("09_mi")
if is_done("09_mi"):
    MI = load_pt(d9 / "mi.pt")
    log("09_mi: loaded")
else:
    t0 = time.time()
    counts, ylab = harvest_counts(MODEL, VAL12)
    bins = CFG["analysis"]["mi_bins"]
    disc = quantile_bins(counts, bins)
    mi = mutual_information(disc, ylab, bins, 12)
    g = torch.Generator().manual_seed(CFG["seed"] + 9)
    null = torch.stack([mutual_information(disc, ylab[torch.randperm(len(ylab), generator=g)], bins, 12)
                        for _ in range(CFG["analysis"]["mi_shuffles"])])
    per_class = torch.stack([mutual_information(disc, (ylab == c).long(), bins, 2) for c in range(12)])
    MI = {"mi": mi.float(), "per_class": per_class.float(), "null_max": float(null.max()),
          "null_p95": float(torch.quantile(null.flatten().float(), 0.95)), "n_above_null": int((mi > null.max()).sum()),
          "bins": bins, "counts": counts.clamp(max=255).to(torch.uint8), "labels": ylab}
    save_pt(MI, d9 / "mi.pt")
    mark_done("09_mi", seconds=round(time.time() - t0))
    log(f"09_mi: done in {time.time() - t0:.0f} s")

In [ ]:
# ---- CHECK 9: the estimator on known cases, then what it says about the network
gen = torch.Generator().manual_seed(0)
yt = torch.arange(4000) % 4                                                        # 4 equal classes
perfect = yt.float()[:, None] + 0.01 * torch.randn(4000, 1, generator=gen)       # carries the label exactly
indep = torch.randn(4000, 1, generator=gen)                                        # carries nothing
mi_p = float(mutual_information(quantile_bins(perfect, 8), yt, 8, 4)[0])
mi_i = float(mutual_information(quantile_bins(indep, 8), yt, 8, 4)[0])
check("09 MI estimator: a perfect feature gives H(Y) = ln 4", abs(mi_p - math.log(4)) < 0.02, f"{mi_p:.4f} vs {math.log(4):.4f}")
check("09 MI estimator: an independent feature gives ~0 (only the small upward bias)", mi_i < 0.02,
      f"{mi_i:.4f} nats; expected bias ~ (8-1)(4-1)/(2*4000) = {21 / 8000:.4f}")

mi, pcm = MI["mi"].double(), MI["per_class"].double()
counts = MI["counts"].float()
dead = int((counts.std(0) == 0).sum())
print(f"H(Y) = {math.log(12):.3f} nats | best neuron {float(mi.max()):.3f} ({float(mi.max()) / math.log(12):.0%}) | "
      f"mean {float(mi.mean()):.3f}")
print(f"shuffle null: p95 {MI['null_p95']:.4f}, max {MI['null_max']:.4f} | neurons above the null max: "
      f"{MI['n_above_null']} of {N_H} | constant (never-firing) neurons: {dead}")
check("09 real class information present (best neuron beats the shuffle null)", float(mi.max()) > MI["null_max"])
print("mean MI per layer:", [round(float(mi[s].mean()), 3) for s in layer_sl])
n_clips = counts.shape[0]
bias = (MI["bins"] - 1) * (2 - 1) / (2 * n_clips)
kth = torch.sort(pcm, dim=1, descending=True).values[:, min(K, N_H) - 1]
print(f"one-vs-rest plug-in bias ~ {bias:.5f} nats; the {K}th-ranked neuron per class is {float(kth.min() / bias):.0f}x "
      f"to {float(kth.max() / bias):.0f}x that bias")
check(f"09 every class's {K}th MI neuron is well above the estimator bias (> 5x)", float(kth.min()) > 5 * bias,
      f"worst class {float(kth.min() / bias):.1f}x", real_data_only=True)

nv_mean = FISH["naive"].double().mean(0)
print(f"\nSpearman(global MI, class-mean naive Fisher): overall {spearman(mi, nv_mean):+.3f}; within layers "
      + ", ".join(f"L{l} {spearman(mi[s], nv_mean[s]):+.3f}" for l, s in enumerate(layer_sl)))
print("  (a negative overall value with weak within-layer values means the two disagree mostly about DEPTH)")

ylab = MI["labels"]
bins_rows = []
for b2 in [4, 16]:
    pc2 = torch.stack([mutual_information(quantile_bins(counts, b2), (ylab == c).long(), b2, 2) for c in range(12)])
    ov = np.mean([len(set(torch.argsort(pc2[c], descending=True)[:K].tolist())
                      & set(torch.argsort(pcm[c], descending=True)[:K].tolist())) / K for c in range(12)])
    bins_rows.append({"bins": b2, "top_k_overlap_with_8_bins": float(ov)})
    print(f"bins {b2:2d}: per-class top-{K} groups overlap with the 8-bin groups by {ov:.0%}")
save_json({"mi_max": float(mi.max()), "mi_mean": float(mi.mean()), "null_max": MI["null_max"], "null_p95": MI["null_p95"],
           "n_above_null": MI["n_above_null"], "dead": dead, "mean_mi_per_layer": [float(mi[s].mean()) for s in layer_sl],
           "spearman_mi_fisher": spearman(mi, nv_mean),
           "spearman_within_layers": [spearman(mi[s], nv_mean[s]) for s in layer_sl],
           "bins_sensitivity": bins_rows, "one_vs_rest_bias": bias}, d9 / "mi_summary.json")

## 10 · Neuron groups and their overlap

Each class gets the top 40 neurons by each method: one-vs-rest **MI**, **naive Fisher** and **Omega**. "Top 40" is a fixed budget, not a threshold. The same budget for every method keeps the comparison fair.

The **MI union** is the set of all neurons in any class's MI group (319 in the original project). It is the default mask in stage 12.

Saved: `10_groups/groups.json`, with group ids, neuron names (`L2N417`, ...) and overlap statistics.

**Check:** the table the defence guide quotes, recomputed: distinct neurons, mean pairwise Jaccard (two random 40-sets give about 0.013), picks per layer, and how many neurons sit in every class's group.

In [ ]:
def topk_per_class(scores, k):
    return {c: torch.argsort(scores[c], descending=True)[:k].tolist() for c in range(scores.shape[0])}


def group_stats(groups):
    sets = [set(g) for g in groups.values()]
    mult = Counter(x for s in sets for x in s)
    jac = [len(a & b) / len(a | b) for i, a in enumerate(sets) for b in sets[i + 1:]]
    picks = Counter(MODEL.layer_of(x) for s in sets for x in s)
    distinct = Counter(MODEL.layer_of(x) for x in mult)
    return {"distinct": len(mult), "mean_jaccard": float(np.mean(jac)),
            "picks_per_layer": [picks.get(l, 0) for l in range(L_H)],
            "distinct_per_layer": [distinct.get(l, 0) for l in range(L_H)],
            "in_all_groups": sum(v == len(sets) for v in mult.values()), "max_multiplicity": max(mult.values())}


d10 = stage_dir("10_groups")
GROUPS = {"mi": topk_per_class(MI["per_class"], K), "fisher": topk_per_class(FISH["naive"], K),
          "omega": topk_per_class(FISH["omega"], K)}
MI_UNION = sorted(set().union(*[set(g) for g in GROUPS["mi"].values()]))
STATS = {m: group_stats(g) for m, g in GROUPS.items()}
p_rand = K * K / N_H
STATS["random_jaccard_expected"] = p_rand / (2 * K - p_rand)
save_json({"k": K, "groups": {m: {CLASSES_12[c]: g for c, g in gr.items()} for m, gr in GROUPS.items()},
           "names": {m: {CLASSES_12[c]: [MODEL.name(x) for x in g] for c, g in gr.items()} for m, gr in GROUPS.items()},
           "mi_union": MI_UNION, "stats": STATS}, d10 / "groups.json")
if not is_done("10_groups"):
    mark_done("10_groups")

In [ ]:
# ---- CHECK 10: the overlap table
print(f"{'method':<8}{'distinct':>9}{'Jaccard':>9}{'in all 12':>10}   picks per layer   distinct per layer")
for m in ["omega", "fisher", "mi"]:
    s = STATS[m]
    print(f"{m:<8}{s['distinct']:>9}{s['mean_jaccard']:>9.3f}{s['in_all_groups']:>10}   {str(s['picks_per_layer']):<17} {s['distinct_per_layer']}")
print(f"random {K}-sets would overlap at Jaccard {STATS['random_jaccard_expected']:.3f}")
print(f"MI union: {len(MI_UNION)} neurons, per layer {STATS['mi']['distinct_per_layer']}")
print("example MI group, class 'right':", [MODEL.name(x) for x in GROUPS["mi"][WORDS.index("right")][:10]])
check("10 groups are valid neuron ids", all(0 <= x < N_H for gr in GROUPS.values() for g in gr.values() for x in g))
check("10 MI union is non-empty and leaves plastic neurons", 0 < len(MI_UNION) < N_H, f"{len(MI_UNION)} of {N_H}")

## 11 · Ablation and sufficiency (held-out test split)

The groups were chosen on **validation**. Every test here runs on the **test** split, so selection and testing never share data. A silenced neuron emits no spikes and holds no membrane, for the whole clip.

- **Per-class ablation:** silence class c's group and measure accuracy per class. Specificity = drop on c − mean drop on the other 11. This is done for MI, naive Fisher and Omega, against 5 random 40-neuron sets.
- **Layer ablation:** 40 neurons from one layer at a time: random, top Omega, top naive Fisher, top unique fraction and top MI within that layer.
- **k-sweep:** how damage grows with lesion size for random, top Omega, top Fisher and top MI.
- **No decision:** when a lesion leaves the readout without spikes, every logit is 0. Such clips count as wrong instead of as "yes" (class 0). In the original project's keep-only table, this tie artefact gave "yes" a recall of 0.988 with a false-positive rate of 0.943.
- **Sufficiency (keep-only):** keep layer 0, the readout and only class c's MI group in layers 1–2. Recall alone is meaningless, because a net that says "c" to everything has 100% recall, so the false-positive rate and precision are reported too. A union keep-only run is compared with a random set of the same size.

**Check:** the unablated baseline equals the stage-7 test accuracy, then the tables: specificity per method, layer ablation, the k-sweep and keep-only sufficiency.

Saved: `11_ablation/ablation.json`. On real data this is about 120 test-set evaluations; my estimate is 10–15 minutes on a GPU (not measured).

In [ ]:
d11 = stage_dir("11_ablation")
REPS = CFG["analysis"]["random_reps"]


def ablate(gids):
    return evaluate(MODEL, TEST12, masks=MODEL.masks_from_gids(gids))


def drops_vs(base_pc, ev, c):
    d = np.array(base_pc) - np.array(ev["per_class"])
    tgt = float(d[c])
    oth = float((d.sum() - d[c]) / (len(d) - 1))
    return tgt, oth, tgt - oth


if is_done("11_ablation"):
    ABL = load_json(d11 / "ablation.json")
    log("11_ablation: loaded")
else:
    t0 = time.time()
    rng = np.random.default_rng(CFG["seed"] + 11)
    base = evaluate(MODEL, TEST12)
    rand = [ablate(rng.choice(N_H, K, replace=False).tolist())["acc"] for _ in range(REPS)]
    rows = []
    for method in ["mi", "fisher", "omega"]:
        for c in range(12):
            ev = ablate(GROUPS[method][c])
            tgt, oth, spec = drops_vs(base["per_class"], ev, c)
            rows.append({"method": method, "class": CLASSES_12[c], "overall": ev["acc"], "drop": base["acc"] - ev["acc"],
                         "target_drop": tgt, "others_drop": oth, "specificity": spec, "no_decision": ev["n_tied"]})
        log(f"  11: {method} groups ablated ({time.time() - t0:.0f} s)")
    om_m, nv_m, uq_m, mi_g = FISH["omega"].mean(0), FISH["naive"].mean(0), FISH["unique"].mean(0), MI["mi"]
    layer_rows = []
    for l in range(L_H):
        lo, hi = MODEL.offsets[l], MODEL.offsets[l + 1]
        k = min(K, hi - lo)
        dr = [base["acc"] - ablate((lo + rng.choice(hi - lo, k, replace=False)).tolist())["acc"] for _ in range(REPS)]
        layer_rows.append({"layer": l, "rule": "random", "drop": float(np.mean(dr)), "drop_std": float(np.std(dr))})
        for rule, score in [("top Omega", om_m), ("top naive Fisher", nv_m), ("top unique fraction", uq_m), ("top MI", mi_g)]:
            g = (lo + torch.argsort(score[lo:hi], descending=True)[:k]).tolist()
            layer_rows.append({"layer": l, "rule": rule, "drop": base["acc"] - ablate(g)["acc"]})
    log(f"  11: layer ablation done ({time.time() - t0:.0f} s)")
    ks_rows = []
    for k in [k for k in CFG["analysis"]["ksweep"] if k <= N_H]:
        dr = [base["acc"] - ablate(rng.choice(N_H, k, replace=False).tolist())["acc"] for _ in range(REPS)]
        row = {"k": k, "random": float(np.mean(dr))}
        for rule, score in [("omega", om_m), ("fisher", nv_m), ("mi", mi_g)]:
            row[rule] = base["acc"] - ablate(torch.argsort(score, descending=True)[:k].tolist())["acc"]
        ks_rows.append(row)
    log(f"  11: k-sweep done ({time.time() - t0:.0f} s)")
    layer0 = list(range(MODEL.offsets[0], MODEL.offsets[1]))
    suff = []
    for c in range(12):
        g = [x for x in GROUPS["mi"][c] if x >= MODEL.offsets[1]]
        ev = evaluate(MODEL, TEST12, masks=MODEL.masks_from_gids(g, keep=True, always_on=layer0))
        cm = np.array(ev["confusion"])
        tot = cm.sum(1) + np.array(ev["tied_per_class"])          # every clip of each class, decided or not
        tp, pred_c, n_c = cm[c, c], cm[:, c].sum(), tot[c]
        suff.append({"class": CLASSES_12[c], "kept": len(g), "recall": tp / max(n_c, 1),
                     "baseline_recall": base["per_class"][c],
                     "false_pos_rate": (pred_c - tp) / max(tot.sum() - n_c, 1), "precision": tp / max(pred_c, 1),
                     "no_decision": ev["n_tied"]})
    deep_union = [x for x in MI_UNION if x >= MODEL.offsets[1]]
    union_acc = evaluate(MODEL, TEST12, masks=MODEL.masks_from_gids(deep_union, keep=True, always_on=layer0))["acc"]
    rand_deep = (MODEL.offsets[1] + rng.choice(N_H - MODEL.offsets[1], len(deep_union), replace=False)).tolist()
    rand_acc = evaluate(MODEL, TEST12, masks=MODEL.masks_from_gids(rand_deep, keep=True, always_on=layer0))["acc"]
    ABL = {"baseline": base, "random_acc": rand, "rows": rows, "layer_rows": layer_rows, "ksweep": ks_rows,
           "sufficiency": suff, "union_keep_only": {"kept_deep": len(deep_union), "acc": union_acc, "random_same_size_acc": rand_acc}}
    save_json(ABL, d11 / "ablation.json")
    mark_done("11_ablation", seconds=round(time.time() - t0))
    log(f"11_ablation: done in {time.time() - t0:.0f} s")

In [ ]:
# ---- CHECK 11: the ablation tables
base_acc = ABL["baseline"]["acc"]
check("11 ablation baseline equals the stage-7 test accuracy", abs(base_acc - BASE_EVAL["testing"]["acc"]) < 0.005,
      f"{base_acc:.4f} vs {BASE_EVAL['testing']['acc']:.4f}")
rnd = np.array(ABL["random_acc"])
print(f"baseline {base_acc:.4f} | {REPS} random {K}-neuron lesions: {rnd.mean():.4f} +- {rnd.std():.4f} "
      f"(drop {100 * (base_acc - rnd.mean()):.1f} pts)")
print(f"\n{'method':<8}{'own class':>11}{'others':>9}{'specificity':>13}{'overall':>9}   (drops in points)")
SPEC = {}
for m in ["mi", "fisher", "omega"]:
    rs = [r for r in ABL["rows"] if r["method"] == m]
    SPEC[m] = float(np.mean([r["specificity"] for r in rs]))
    print(f"{m:<8}{100 * np.mean([r['target_drop'] for r in rs]):>11.1f}{100 * np.mean([r['others_drop'] for r in rs]):>9.1f}"
          f"{SPEC[m]:>+13.3f}{100 * np.mean([r['drop'] for r in rs]):>9.1f}")
print(f"\nown-class drop per class (points):\n{'class':<12}" + "".join(f"{m:>9}" for m in ["mi", "fisher", "omega"]))
for nm in CLASSES_12:
    vals = [next(r["target_drop"] for r in ABL["rows"] if r["method"] == m and r["class"] == nm) for m in ["mi", "fisher", "omega"]]
    print(f"{nm:<12}" + "".join(f"{100 * v:>9.1f}" for v in vals))
print(f"\nlayer ablation ({K} neurons from one layer), drop in points:")
for l in range(L_H):
    print(f"  layer {l}: " + ", ".join(f"{r['rule']} {100 * r['drop']:.2f}" for r in ABL["layer_rows"] if r["layer"] == l))
print("\nk-sweep, drop in points:")
for r in ABL["ksweep"]:
    print(f"  k={r['k']:4d}: random {100 * r['random']:6.2f} | Omega {100 * r['omega']:6.2f} | Fisher {100 * r['fisher']:6.2f} | MI {100 * r['mi']:6.2f}")
print(f"\nkeep-only (layer 0 + readout + one MI group):\n{'class':<12}{'recall':>8}{'base':>7}{'false-pos':>11}{'precision':>11}"
      f"{'no decision':>13}")
for r in ABL["sufficiency"]:
    print(f"{r['class']:<12}{r['recall']:>8.3f}{r['baseline_recall']:>7.3f}{r['false_pos_rate']:>11.3f}{r['precision']:>11.3f}"
          f"{r.get('no_decision', 0):>13}")
print("('no decision' = test clips whose readout got no spike at all; they count as wrong, not as 'yes')")
u = ABL["union_keep_only"]
print(f"keep layer 0 + all {u['kept_deep']} deep MI-union neurons: {u['acc']:.4f} vs random same size {u['random_same_size_acc']:.4f}")
check("11 MI groups are more class-specific than Fisher groups (specificity)", SPEC["mi"] > SPEC["fisher"],
      f"MI {SPEC['mi']:+.3f}, Fisher {SPEC['fisher']:+.3f}, Omega {SPEC['omega']:+.3f} (a finding, not a bug, if it fails)",
      real_data_only=True)

if HAVE_PLT:
    plt.figure(figsize=(12, 3.2))
    xs = np.arange(12)
    for i, m in enumerate(["mi", "fisher", "omega"]):
        plt.bar(xs + (i - 1) * 0.27, [100 * next(r["target_drop"] for r in ABL["rows"] if r["method"] == m and r["class"] == nm)
                                      for nm in CLASSES_12], width=0.27, label=m)
    plt.xticks(xs, CLASSES_12, rotation=45)
    plt.ylabel("own-class drop (pts)")
    plt.legend()
    plt.title(f"removing each class's top-{K} neurons (test split)")
    savefig("11_ablation_per_class")

## 11b · Groups from a combined MI + Fisher score (held-out test split)

Stage 10 picks each class's group with one measure. Here the two are blended. For class c and neuron i:

score = α · rank(MIᶜᵢ) + (1 − α) · rank(Fᶜᵢᵢ),

where each rank is a percentile in [0, 1], and α runs over `CFG["combo"]["alphas"]` (α = 1 is pure MI, α = 0 pure naive Fisher). Each class keeps its top-K neurons by this score. The ranks come in two flavours:

- **global:** ranked over all hidden neurons. Fisher's depth bias stays in, so a low α pulls the group into layer 0. With these ranks, α = 1 and α = 0 reproduce the stage-10 MI and Fisher groups (a check below).
- **within_layer:** each neuron is ranked only against its own layer. The depth bias is removed, so α = 0 is a "Fisher within layer" selector and the groups spread over every layer.

Every group set gets the stage-11 per-class ablation (same test clips, same baseline), and stage 12 freezes the union of each set's groups in its own arm (`combo_<global|layer>_a<100·α>`). The base network and every continual arm here are trained with BPTT.

**Check:** with global ranks, α = 1 and α = 0 match the MI and Fisher groups; every group has K valid ids. Then the table: union size and layers, overlap with the MI and Fisher groups, and specificity per α.

Saved: `11b_combo/combo.json`.

In [ ]:
d11b = stage_dir("11b_combo")


def rank_scores(score, within_layer):
    """Percentile rank per class (0 = lowest, 1 = highest), over all hidden neurons or within each layer. [C, N]"""
    out = torch.zeros(score.shape, dtype=torch.float64)
    parts = layer_sl if within_layer else [slice(0, score.shape[1])]
    for c in range(score.shape[0]):
        for s in parts:
            r = avg_ranks(score[c, s])
            out[c, s] = r / max(len(r) - 1, 1)
    return out


def combo_name(mode, alpha):
    return f"combo_{'layer' if mode == 'within_layer' else 'global'}_a{int(round(100 * alpha)):03d}"


def mean_overlap(groups, ref):
    return float(np.mean([len(set(groups[c]) & set(ref[c])) / K for c in range(len(ref))]))


# ties (e.g. the best neuron of every layer at rank 1 within layers) are broken by a tiny global-rank term
TIEBREAK = 1e-6 * (rank_scores(MI["per_class"], False) + rank_scores(FISH["naive"], False))
COMBO_GROUPS, COMBO_META = {}, {}
for mode in CFG["combo"]["rank_modes"]:
    r_mi = rank_scores(MI["per_class"], mode == "within_layer")
    r_f = rank_scores(FISH["naive"], mode == "within_layer")
    for a in CFG["combo"]["alphas"]:
        name = combo_name(mode, a)
        COMBO_GROUPS[name] = topk_per_class(a * r_mi + (1 - a) * r_f + TIEBREAK, K)
        COMBO_META[name] = {"mode": mode, "alpha": float(a)}

if is_done("11b_combo"):
    COMBO = load_json(d11b / "combo.json")
    log("11b_combo: loaded")
else:
    t0 = time.time()
    base_pc = ABL["baseline"]["per_class"]
    COMBO = {}
    for name, groups in COMBO_GROUPS.items():
        rows = []
        for c in range(12):
            ev = ablate(groups[c])
            tgt, oth, spec = drops_vs(base_pc, ev, c)
            rows.append({"class": CLASSES_12[c], "target_drop": tgt, "others_drop": oth, "specificity": spec,
                         "no_decision": ev["n_tied"]})
        union = set().union(*[set(g) for g in groups.values()])
        COMBO[name] = {**COMBO_META[name], "specificity": float(np.mean([r["specificity"] for r in rows])),
                       "target_drop": float(np.mean([r["target_drop"] for r in rows])),
                       "others_drop": float(np.mean([r["others_drop"] for r in rows])),
                       "union_size": len(union), "union_per_layer": [sum(MODEL.layer_of(x) == l for x in union) for l in range(L_H)],
                       "overlap_mi": mean_overlap(groups, GROUPS["mi"]), "overlap_fisher": mean_overlap(groups, GROUPS["fisher"]),
                       "stats": group_stats(groups), "rows": rows}
        log(f"  11b: {name} ablated ({time.time() - t0:.0f} s)")
    save_json(COMBO, d11b / "combo.json")
    mark_done("11b_combo", seconds=round(time.time() - t0))
    log(f"11b_combo: done in {time.time() - t0:.0f} s")


In [ ]:
# ---- CHECK 11b: the combined-score groups
check("11b every combined group holds K valid neuron ids",
      all(len(g) == K and all(0 <= x < N_H for x in g) for gs in COMBO_GROUPS.values() for g in gs.values()))
if "global" in CFG["combo"]["rank_modes"] and {0.0, 1.0} <= set(CFG["combo"]["alphas"]):
    o_mi = mean_overlap(COMBO_GROUPS[combo_name("global", 1.0)], GROUPS["mi"])
    o_f = mean_overlap(COMBO_GROUPS[combo_name("global", 0.0)], GROUPS["fisher"])
    check("11b global ranks: alpha = 1 gives the MI groups, alpha = 0 the Fisher groups", o_mi >= 0.95 and o_f >= 0.95,
          f"overlap {o_mi:.0%} with MI, {o_f:.0%} with Fisher (ties in the scores can move a neuron or two)")
print(f"{'group set':<20}{'alpha':>6}{'union':>7}{'union per layer':>17}{'vs MI':>8}{'vs Fisher':>11}"
      f"{'own drop':>10}{'others':>8}{'specificity':>13}   (drops in points)")
for name, r in COMBO.items():
    print(f"{name:<20}{r['alpha']:>6.2f}{r['union_size']:>7}{str(r['union_per_layer']):>17}{r['overlap_mi']:>8.0%}"
          f"{r['overlap_fisher']:>11.0%}{100 * r['target_drop']:>10.1f}{100 * r['others_drop']:>8.1f}{r['specificity']:>+13.3f}")
print(f"stage 11 for reference: specificity MI {SPEC['mi']:+.3f}, Fisher {SPEC['fisher']:+.3f}, Omega {SPEC['omega']:+.3f}")

if HAVE_PLT:
    plt.figure(figsize=(6, 3.5))
    for mode in CFG["combo"]["rank_modes"]:
        rs = sorted((r for r in COMBO.values() if r["mode"] == mode), key=lambda r: r["alpha"])
        plt.plot([r["alpha"] for r in rs], [r["specificity"] for r in rs], marker="o", label=f"{mode} ranks")
    plt.xlabel("alpha (0 = Fisher only, 1 = MI only)")
    plt.ylabel("ablation specificity")
    plt.legend()
    plt.title("combined MI + Fisher groups (test split)")
    savefig("11b_combo_specificity")


## 12 · Forgetting vs plasticity: learning "zero"

Every run here starts from the same base network, adds a 13th output for "zero" and trains it with BPTT. Batches are half "zero" clips and half old clips. The 12 old output rows stay frozen in every run, so runs differ only in **which hidden neurons may change** (their incoming weights frozen or not), **how long** they train, the **learning rate**, and how many old clips carry their **label** (replay):

- **replay fraction 0:** old clips are unlabelled; they only keep the new output below the best old output (softplus term), as in the pipeline notebook's no-replay arms.
- **replay fraction f > 0:** a share f of the whole batch is labelled old clips and gets the 13-way cross-entropy. f = 0.5 labels every old clip, which is the pipeline's full replay.

The helpers below (data sets, the new-row initialisation, masks, scoring) are the pipeline notebook's stage-12 code.

In [ ]:
d12 = stage_dir("12_setup")
(d12 / "arms").mkdir(exist_ok=True)
OLD_TRAIN = make_set("training", drop_new_from_unknown=True, train=True)          # the 12 old classes
NEW_TRAIN = make_set("training", words=False, unknown=False, silence=False, new=True, train=True)
VAL13 = make_set("validation", new=True)
TEST13 = make_set("testing", new=True)
log(f"12 sets: old-class training {len(OLD_TRAIN)} | '{NEW_WORD}' training {len(NEW_TRAIN)} | "
    f"validation {len(VAL13)} | test {len(TEST13)} (13 classes)")

BASE13_TEST = evaluate(MODEL, TEST13, 13)          # the base net cannot answer 12: old-class scores = the baseline
BASE13_VAL = evaluate(MODEL, VAL13, 13)
BASE_OLD = np.array(BASE13_TEST["per_class"][:12])
BASE_OLD_VAL = np.array(BASE13_VAL["per_class"][:12])
zrow = np.array(BASE13_TEST["confusion"][NEW], dtype=float)
zrow /= max(zrow.sum(), 1.0)
top3 = np.argsort(-zrow)[:3]
log(f"  before learning it, the base network files test '{NEW_WORD}' clips as: "
    + ", ".join(f"{CLASSES_13[i]} {zrow[i]:.0%}" for i in top3))
gap = float(np.abs(BASE_OLD - np.array(ABL["baseline"]["per_class"])).max())
check("12 old-class test scores equal the stage-11 baseline (same clips)", gap < 0.01,
      f"largest per-class difference {gap:.4f} (tiny differences can come from different batch composition)")

In [ ]:
NB_INIT, BS_CL = CFG["cl"]["init_batches"], CFG["cl"]["batch_size"]


def readout_trace(spikes_last, beta):
    """xbar[t] = beta * xbar[t-1] + s[t]: what the readout membrane 'sees' of each last-layer neuron."""
    xb, out = torch.zeros_like(spikes_last[0]), []
    for s in spikes_last:
        xb = beta * xb + s
        out.append(xb)
    return torch.stack(out)                          # [T, B, N_last]


@torch.no_grad()
def init_new_row(model):
    """Start the new readout row at f_new - alpha * f_old (mean readout-input traces of 'zero' and of old clips),
    with alpha chosen so that the row still responds POSITIVELY to 'zero' clips, then scale it so its
    max-membrane logit on 'zero' clips matches the best old logit on them.
    Why not a zero row, or plain f_new - f_old: if the new membrane is never positive, its maximum sits at
    t = 0 where nothing has arrived yet, and the gradient is exactly zero (the max-membrane dead zone)."""
    def mean_trace(data):
        tot, n = 0.0, 0
        for wav, _ in data.batches(BS_CL, shuffle=True, seed=7, max_batches=NB_INIT):
            xb = readout_trace(model(make_input(wav), record=True)["spikes"][-1], model.beta)
            tot, n = tot + xb.mean(0).sum(0), n + xb.shape[1]
        return tot / max(n, 1)

    f_new, f_old = mean_trace(NEW_TRAIN), mean_trace(OLD_TRAIN)
    alpha = min(1.0, 0.5 * float(f_new @ f_new) / max(float(f_old @ f_new), 1e-12))
    w = f_new - alpha * f_old                       # w . f_new >= 0.5 |f_new|^2 > 0
    w = w / w.norm().clamp_min(1e-12)
    zn, zo = [], []
    for wav, _ in NEW_TRAIN.batches(BS_CL, shuffle=True, seed=8, max_batches=NB_INIT):
        out = model(make_input(wav), record=True)
        zn.append((readout_trace(out["spikes"][-1], model.beta) @ w).max(0).values.cpu())
        zo.append(out["logits"].max(-1).values.cpu())
    zn, zo = float(torch.cat(zn).median()), float(torch.cat(zo).median())
    target = zo if zo > 0 else 1.0
    scale = target / zn if zn > 0 else 1.0
    return w * scale, {"alpha": alpha, "scale": scale, "median_new_logit_unscaled": zn, "median_best_old_logit": zo}


def frozen_set(kind):
    """Which hidden neurons keep their incoming weights fixed."""
    rng = np.random.default_rng(CFG["seed"] + 1234)
    mi, n = set(MI_UNION), len(MI_UNION)
    fisher_top = set(torch.argsort(FISH["naive"].mean(0), descending=True)[:n].tolist())
    layer0 = set(range(MODEL.offsets[0], MODEL.offsets[1]))
    if kind == "none":
        return set()
    if kind == "all":
        return set(range(N_H))
    if kind == "mi":
        return mi
    if kind == "random":
        return set(rng.choice(N_H, n, replace=False).tolist())
    if kind == "random_layer_matched":
        out = set()
        for l in range(L_H):
            lo, hi = MODEL.offsets[l], MODEL.offsets[l + 1]
            n_l = sum(lo <= g < hi for g in mi)
            out |= set((lo + rng.choice(hi - lo, n_l, replace=False)).tolist())
        return out
    if kind == "fisher":
        return fisher_top
    if kind == "mi+layer0":
        return mi | layer0
    if kind == "mi+fisher":
        return mi | fisher_top
    if kind == "layer0":
        return layer0
    if kind.startswith("combo:"):                   # the union of one stage-11b group set
        return set().union(*[set(g) for g in COMBO_GROUPS[kind.split(":", 1)[1]].values()])
    raise ValueError(kind)


def plasticity_masks(model13, frozen, freeze_old_readout):
    """One 0/1 vector per weight matrix, over its rows (= the incoming weights of one neuron); 1 = may change."""
    masks = []
    for l in range(L_H):
        lo, hi = model13.offsets[l], model13.offsets[l + 1]
        m = torch.ones(hi - lo)
        idx = [g - lo for g in frozen if lo <= g < hi]
        if idx:
            m[idx] = 0.0
        masks.append(m.to(DEVICE))
    r = torch.ones(model13.n_out)
    if freeze_old_readout:
        r[:NEW] = 0.0
    masks.append(r.to(DEVICE))
    return masks


def attach_masks(model13, masks):
    return [lin.weight.register_hook(lambda g, m=m: g.masked_fill(m[:, None] == 0, 0.0)) for lin, m in zip(model13.fc, masks)]


def cl_batches(seed):
    """Half new-word clips, half old-class clips. One pass over the new-word clips = one epoch."""
    gen = torch.Generator().manual_seed(seed)
    half = max(1, BS_CL // 2)
    new_order = torch.randperm(len(NEW_TRAIN), generator=gen)
    old_order = torch.randperm(len(OLD_TRAIN), generator=gen)
    for b in range(math.ceil(len(NEW_TRAIN) / half)):
        ni = new_order[b * half:(b + 1) * half]
        oi = old_order[torch.arange(b * half, b * half + len(ni)) % len(OLD_TRAIN)]
        wav = torch.cat([NEW_TRAIN.get(ni, DEVICE, AUG, gen), OLD_TRAIN.get(oi, DEVICE, AUG, gen)])
        yield wav, torch.cat([NEW_TRAIN.labels[ni], OLD_TRAIN.labels[oi]]).to(DEVICE)


def cl_loss(logits, y, replay):
    """replay: 13-way cross-entropy with old labels. No replay: CE on new-word clips + keep the new
    logit below the best old logit on old clips (their labels are never used)."""
    if replay:
        return F.cross_entropy(logits, y)
    is_new = y == NEW
    loss = logits.new_zeros(())
    if bool(is_new.any()):
        loss = loss + F.cross_entropy(logits[is_new], y[is_new])
    if bool((~is_new).any()):
        z = logits[~is_new]
        loss = loss + F.softplus(z[:, NEW] - z[:, :NEW].max(-1).values).mean()
    return loss


def local_error(logits, y):
    """d(loss)/d(new logit) for each clip under the no-replay objective."""
    p_new = torch.softmax(logits, -1)[:, NEW]
    gap = logits[:, NEW] - logits[:, :NEW].max(-1).values
    return torch.where(y == NEW, p_new - 1.0, torch.sigmoid(gap))


@torch.no_grad()
def logits_of(model, data):
    """Logits [N, n_out] and labels [N] of a ClipSet, in its fixed order, on the CPU."""
    model.eval()
    Z, Y = [], []
    for wav, y in data.batches(CFG["train"]["eval_batch_size"]):
        Z.append(model(make_input(wav))["logits"].float().cpu())
        Y.append(y.cpu())
    return torch.cat(Z), torch.cat(Y)


def class_scores(pred, y, n):
    """Per-class accuracy [n] and confusion matrix [true, predicted] as NumPy arrays.
    pred = -1 (no decision, see decide) counts as wrong and is left out of the confusion matrix."""
    ok = pred >= 0
    conf = torch.bincount(y[ok] * n + pred[ok], minlength=n * n).view(n, n).double()
    total = torch.bincount(y, minlength=n).double()
    return (conf.diagonal() / total.clamp_min(1)).numpy(), conf.numpy()


def auc(pos, neg):
    """P(score of a random positive > score of a random negative); ties count 1/2 (Mann-Whitney U)."""
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    r = avg_ranks(torch.cat([pos, neg]))
    return float((r[:len(pos)].sum() - len(pos) * (len(pos) - 1) / 2) / (len(pos) * len(neg)))


def summarize(Z, Y, base_old):
    """Scores of one 13-output model from its logits.
    13-way: the decision as trained (ties = no decision). drift: old clips decided among the 12 old outputs
    only. new_word_auc: how well the margin z_new - max z_old separates new-word clips from old clips."""
    pred = decide(Z)
    pc, conf = class_scores(pred, Y, 13)
    n_new, n_old = max(int((Y == NEW).sum()), 1), max(int((Y < NEW).sum()), 1)
    forget = 100.0 * (base_old - pc[:12])
    zr = conf[NEW] / n_new
    wrong = zr.copy()
    wrong[NEW] = -1.0
    old = Y < NEW
    pc12, _ = class_scores(decide(Z[old, :NEW]), Y[old], 12)
    drift = 100.0 * (base_old - pc12)
    g = Z[:, NEW] - Z[:, :NEW].max(1).values
    return {"acc_13": float(np.trace(conf) / max(len(Y), 1)), "no_decision": int((pred < 0).sum()),
            "old_mean_acc": float(pc[:12].mean()), "forgetting_mean_pts": float(forget.mean()),
            "forgetting_pts": dict(zip(CLASSES_12, forget.round(2).tolist())),
            "worst_class": CLASSES_12[int(forget.argmax())], "worst_forgetting_pts": float(forget.max()),
            "new_word_acc": float(pc[NEW]), "new_word_to_unknown": float(zr[UNKNOWN]),
            "new_word_top_confusion": CLASSES_13[int(wrong.argmax())],
            "old_clips_called_new": float(conf[:12, NEW].sum() / n_old),
            "per_class": dict(zip(CLASSES_13, pc.round(4).tolist())),
            "drift_forgetting_mean_pts": float(drift.mean()),
            "drift_forgetting_pts": dict(zip(CLASSES_12, drift.round(2).tolist())),
            "new_word_auc": auc(g[Y == NEW], g[old]),
            "new_word_auc_vs_class": {CLASSES_12[c]: auc(g[Y == NEW], g[Y == c]) for c in range(12)}}


def epoch_row(model13, ep, losses, name):
    s = summarize(*logits_of(model13, VAL13), BASE_OLD_VAL)
    row = {"epoch": ep + 1, "loss": float(np.mean(losses)) if losses else None,
           "val_drift_pts": s["drift_forgetting_mean_pts"], "val_new_word_auc": s["new_word_auc"],
           "val_forgetting_pts": s["forgetting_mean_pts"], "val_new_word_acc": s["new_word_acc"]}
    log(f"    {name} ep {ep + 1}: loss {row['loss'] if row['loss'] is None else round(row['loss'], 3)} | val drift "
        f"{s['drift_forgetting_mean_pts']:+.2f} pts, '{NEW_WORD}' AUC {s['new_word_auc']:.3f} | 13-way: forgetting "
        f"{s['forgetting_mean_pts']:+.2f}, '{NEW_WORD}' acc {s['new_word_acc']:.3f}")
    return row

In [ ]:
# ---- the arms
BPTT_ARMS = {                       # name: (frozen hidden neurons, replay?)
    "readout_only": ("all", False),
    "no_mask": ("none", False),
    "mi_mask": ("mi", False),
    "random_mask": ("random", False),
    "random_layer_matched": ("random_layer_matched", False),
    "fisher_mask": ("fisher", False),
    "mi_plus_layer0": ("mi+layer0", False),
    "mi_plus_fisher": ("mi+fisher", False),
    "mi_mask_replay": ("mi", True),
    "no_mask_replay": ("none", True),
    "layer0_only": ("layer0", False),
}
for _name in COMBO_GROUPS:                           # one arm per stage-11b group set
    BPTT_ARMS[_name] = ("combo:" + _name, False)


def run_bptt_arm(name):
    kind, replay = BPTT_ARMS[name]
    seed_all(CFG["seed"] + 77)
    model13 = expand_readout(MODEL, W_NEW0)
    frozen = frozen_set(kind)
    masks = plasticity_masks(model13, frozen, freeze_old_readout=True)       # the same in every arm
    hooks = attach_masks(model13, masks)
    ref = [lin.weight.detach().clone() for lin in model13.fc]
    opt = torch.optim.Adam(model13.parameters(), lr=CFG["cl"]["lr"], weight_decay=0.0)
    curve = []
    for ep in range(CFG["cl"]["epochs"]):
        model13.train()
        losses = []
        for wav, y in cl_batches(seed=1000 + ep):
            loss = cl_loss(model13(make_input(wav))["logits"], y, replay)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        curve.append(epoch_row(model13, ep, losses, name))
    for h in hooks:
        h.remove()
    moved = max(float(((lin.weight.detach() - r).abs() * (1 - m)[:, None]).max()) for lin, r, m in zip(model13.fc, ref, masks))
    check(f"12 {name}: frozen weights did not move", moved == 0.0, f"largest change of a frozen weight {moved:.1e}")
    per_layer = [sum(model13.offsets[l] <= g < model13.offsets[l + 1] for g in frozen) for l in range(L_H)]
    return model13, curve, {"frozen_hidden": len(frozen), "frozen_per_layer": per_layer, "replay": replay,
                            "old_readout_frozen": True}


@torch.no_grad()
def run_local_rule_arm():
    """Backprop-free: only the new readout row learns, by dW = -lr * err * xbar(t*), where err is the error
    at the new output and xbar(t*) is the presynaptic trace when the new output peaked."""
    model13 = expand_readout(MODEL, W_NEW0)
    W = model13.fc[-1].weight
    lr, norm_ref, curve = CFG["cl"]["local_lr"], None, []
    for ep in range(CFG["cl"]["epochs"]):
        for wav, y in cl_batches(seed=2000 + ep):
            out = model13(make_input(wav), record=True)
            traces = readout_trace(out["spikes"][-1], model13.beta)               # [T, B, N]
            t_star = out["mem_out"][:, :, NEW].argmax(0)                            # [B]
            pre = traces[t_star, torch.arange(len(y), device=DEVICE)]                # [B, N]
            err = local_error(out["logits"], y)
            if norm_ref is None:                                                     # normalised LMS step size
                norm_ref = float((pre ** 2).sum(1).mean().clamp_min(1e-6))
            W[NEW] -= (lr / norm_ref) * (err[:, None] * pre).mean(0)
        curve.append(epoch_row(model13, ep, [], "local_rule"))
    return model13, curve, {"lr": lr, "norm_ref": norm_ref}


def local_rule_vs_autograd(model13):
    """The local rule should equal the true gradient of the no-replay objective for the new row."""
    wav, y = next(cl_batches(seed=31))
    out = model13(make_input(wav), record=True)
    z = out["logits"]
    per_clip = torch.where(y == NEW, -torch.log_softmax(z, -1)[:, NEW], F.softplus(z[:, NEW] - z[:, :NEW].max(-1).values))
    g_auto = torch.autograd.grad(per_clip.mean(), model13.fc[-1].weight)[0][NEW]
    with torch.no_grad():
        traces = readout_trace([s.detach() for s in out["spikes"][-1]], model13.beta)
        t_star = out["mem_out"][:, :, NEW].argmax(0)
        pre = traces[t_star, torch.arange(len(y), device=DEVICE)]
        g_local = (local_error(z.detach(), y)[:, None] * pre).mean(0)
    return float(F.cosine_similarity(g_auto, g_local, dim=0)), float((g_auto - g_local).abs().max())


@torch.no_grad()
def hidden_features(model, data):
    X, Y = [], []
    for wav, y in data.batches(CFG["train"]["eval_batch_size"]):
        X.append(torch.cat(model(make_input(wav))["counts"], 1).cpu())
        Y.append(y.cpu())
    return torch.cat(X).double(), torch.cat(Y)


def slda_fit(X, y, n_classes, shrink):
    """Deep SLDA (Hayes & Kanan 2020), batch form: W_k = P mu_k, b_k = -mu_k.P.mu_k / 2,
    P = (Sigma + shrink * mean(diag) * I)^-1, Sigma = pooled within-class covariance. No backprop."""
    D = X.shape[1]
    mu = torch.stack([X[y == c].mean(0) if bool((y == c).any()) else torch.zeros(D, dtype=X.dtype) for c in range(n_classes)])
    R = X - mu[y]
    S = R.T @ R / max(len(X) - n_classes, 1)
    lam = shrink * float(S.diagonal().mean().clamp_min(1e-12))
    P = torch.linalg.inv(S + lam * torch.eye(D, dtype=X.dtype))
    W = mu @ P
    return W, -0.5 * (W * mu).sum(1)


class SLDAModel(nn.Module):
    """The frozen spiking network + an SLDA head on its spike counts."""

    def __init__(self, base, W, b, cols):
        super().__init__()
        self.base, self.cols = base, cols
        self.register_buffer("W", W.float())
        self.register_buffer("b", b.float())

    def forward(self, x, masks=None, record=False):
        out = self.base(x)
        f = torch.cat(out["counts"], 1)[:, self.cols]
        return {"logits": f @ self.W.T + self.b, "counts": out["counts"]}


def run_slda_arms():
    t0 = time.time()
    Xo, yo = hidden_features(MODEL, OLD_TRAIN)            # statistics of the old classes (one pass)
    Xn, yn = hidden_features(MODEL, NEW_TRAIN)            # ... and of the new word
    X, yy = torch.cat([Xo, Xn]), torch.cat([yo, yn])
    shrink = CFG["cl"]["slda_shrinkage"]
    out = {}
    for feat, cols in [("last", slice(MODEL.offsets[-2], MODEL.offsets[-1])), ("all", slice(0, N_H))]:
        W12, b12 = slda_fit(Xo[:, cols], yo, 12, shrink)
        W13, b13 = slda_fit(X[:, cols], yy, 13, shrink)
        old12 = float(np.mean(evaluate(SLDAModel(MODEL, W12, b12, cols).to(DEVICE), TEST13, 13)["per_class"][:12]))
        out["slda_" + feat] = (SLDAModel(MODEL, W13, b13, cols).to(DEVICE), [],
                               {"features": feat, "dim": int(W13.shape[1]), "slda_12class_old_mean_acc": old12,
                                "trained_readout_old_mean_acc": float(BASE_OLD.mean()), "seconds": round(time.time() - t0)})
    return out


class OneWayNewNeurons(nn.Module):
    """The frozen base network + k new LIF neurons beside its last hidden layer. The new neurons read the
    frozen layer below (old -> new allowed) and feed only the new output (new -> old forbidden), so the old
    logits are exactly the base network's."""

    def __init__(self, base, k_new, w_old_init):
        super().__init__()
        self.base = base
        n_prev = base.hidden[-2]
        self.fc_new = nn.Linear(n_prev, k_new, bias=False)
        nn.init.normal_(self.fc_new.weight, 0.0, math.sqrt(2.0 / n_prev))
        self.w_old = nn.Parameter(w_old_init.clone().float())        # new output <- old last-layer neurons
        self.w_new = nn.Parameter(torch.zeros(k_new))                 # new output <- new neurons

    def trainable(self):
        return [self.fc_new.weight, self.w_old, self.w_new]

    def forward(self, x, masks=None, record=False):
        with torch.no_grad():
            out = self.base(x, record=True)                            # the old network, untouched
        s_prev, s_last = out["spikes"][-2], out["spikes"][-1]
        B, k = x.shape[1], self.w_new.numel()
        mem, cnt = x.new_zeros(B, k), x.new_zeros(B, k)
        v, vs = x.new_zeros(B), []
        for t in range(x.shape[0]):
            s_new, mem = lif_step(mem, self.fc_new(s_prev[t]), self.base.beta, self.base.thr)
            cnt = cnt + s_new.detach()
            v = self.base.beta * v + s_last[t] @ self.w_old + s_new @ self.w_new
            vs.append(v)
        best = torch.stack(vs).max(0).values
        return {"logits": torch.cat([out["logits"], best[:, None]], 1), "counts": out["counts"] + [cnt]}


def run_one_way_arm():
    seed_all(CFG["seed"] + 78)
    ow = OneWayNewNeurons(MODEL, CFG["cl"]["k_new_neurons"], W_NEW0).to(DEVICE)
    before = {k: v.detach().clone() for k, v in MODEL.state_dict().items()}
    opt = torch.optim.Adam(ow.trainable(), lr=CFG["cl"]["lr"])
    curve = []
    for ep in range(CFG["cl"]["epochs"]):
        losses = []
        for wav, y in cl_batches(seed=3000 + ep):
            loss = cl_loss(ow(make_input(wav))["logits"], y, replay=False)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        curve.append(epoch_row(ow, ep, losses, "one_way_new_neurons"))
    check("12 one_way_new_neurons: base network bit-for-bit unchanged",
          all(torch.equal(before[k], v) for k, v in MODEL.state_dict().items()))
    wav, _ = next(TEST13.batches(32))
    x = make_input(wav)
    with torch.no_grad():
        same = torch.equal(ow(x)["logits"][:, :12], MODEL(x)["logits"])
    check("12 one_way_new_neurons: old logits identical to the base network's", same)
    with torch.no_grad():
        rate = float(ow(x)["counts"][-1].mean()) / T_STEPS
    return ow, curve, {"k_new": CFG["cl"]["k_new_neurons"], "new_neuron_rate": rate}


# ---- forward-forward arms: the hidden layers learn without backprop, the new row by the local rule
FF_ARMS = {                         # name: (negative data, frozen hidden neurons)
    "ff_contrast": ("old", "none"),
    "ff_contrast_layer0_frozen": ("old", "layer0"),
    "ff_hybrid": ("hybrid", "none"),
}


def ff_counts(model, x):
    """The normal forward pass, except that every layer reads its input DETACHED and its own membrane from the
    previous step DETACHED. Spike values are unchanged; a spike's gradient now reaches only the weights into that
    neuron, at that step (no backprop across layers or through time). Returns per-layer spike counts [B, N_l]."""
    T, B, _ = x.shape
    mems = [x.new_zeros(B, n) for n in model.hidden]
    counts = [x.new_zeros(B, n) for n in model.hidden]
    for t in range(T):
        h = x[t]
        for l in range(len(model.hidden)):
            s, mems[l] = lif_step(mems[l].detach(), model.fc[l](h.detach()), model.beta, model.thr)
            counts[l] = counts[l] + s
            h = s
    return counts


def ff_goodness(counts):
    """Goodness of each layer for each clip: mean over its neurons of the squared firing rate. [L, B]"""
    return torch.stack([(c / T_STEPS).pow(2).mean(1) for c in counts])


def hybrid_negatives(x, seed):
    """Negative data in time (Hinton's hybrid images): each clip's spike train is spliced with another clip's in
    random blocks of `hybrid_block` steps, so local structure is real but the whole is no word."""
    T, B, _ = x.shape
    blk = CFG["cl"]["ff"]["hybrid_block"]
    g = torch.Generator().manual_seed(seed)
    own = (torch.rand(math.ceil(T / blk), B, generator=g) < 0.5).repeat_interleave(blk, 0)[:T].to(x.device)
    shift = int(torch.randint(1, max(B, 2), (1,), generator=g))
    partner = (torch.arange(B, device=x.device) + shift) % B
    return torch.where(own[:, :, None], x, x[:, partner])


def ff_split(x, y, neg, seed):
    """Positive and negative inputs of one batch. 'old': positives = new-word clips, negatives = old clips (their
    labels are never used). 'hybrid': positives = every real clip, negatives = spliced clips (no labels at all)."""
    if neg == "old":
        return x[:, y == NEW], x[:, y != NEW]
    return x, hybrid_negatives(x, seed)


def ff_loss(model, x, y, neg, theta, seed):
    """Sum over layers of each layer's own loss; the layers share no gradient."""
    pos, negx = ff_split(x, y, neg, seed)
    k, th = CFG["cl"]["ff"]["kappa"], theta[:, None]
    zp = k * (ff_goodness(ff_counts(model, pos)) - th) / th
    zn = k * (ff_goodness(ff_counts(model, negx)) - th) / th
    return (F.softplus(-zp).mean(1) + F.softplus(zn).mean(1)).sum()


@torch.no_grad()
def ff_thresholds(model, neg):
    """Per-layer theta = the base network's mean goodness on positives and on negatives, averaged."""
    gp, gn = [], []
    for b, (wav, y) in enumerate(cl_batches(seed=4999)):
        if b >= NB_INIT:
            break
        pos, negx = ff_split(make_input(wav), y, neg, seed=b)
        gp.append(ff_goodness(ff_counts(model, pos)).mean(1))
        gn.append(ff_goodness(ff_counts(model, negx)).mean(1))
    return (0.5 * (torch.stack(gp).mean(0) + torch.stack(gn).mean(0))).clamp_min(1e-8)


def run_ff_arm(name):
    neg, kind = FF_ARMS[name]
    seed_all(CFG["seed"] + 79)
    model13 = expand_readout(MODEL, W_NEW0)
    frozen = frozen_set(kind)
    masks = plasticity_masks(model13, frozen, freeze_old_readout=True)
    hooks = attach_masks(model13, masks)
    ref = [lin.weight.detach().clone() for lin in model13.fc]
    theta = ff_thresholds(model13, neg)
    opt = torch.optim.Adam([lin.weight for lin in model13.fc[:-1]], lr=CFG["cl"]["lr"], weight_decay=0.0)
    W, lr_loc, norm_ref, curve = model13.fc[-1].weight, CFG["cl"]["local_lr"], None, []
    for ep in range(CFG["cl"]["epochs"]):
        model13.train()
        losses = []
        for b, (wav, y) in enumerate(cl_batches(seed=4000 + ep)):
            x = make_input(wav)
            loss = ff_loss(model13, x, y, neg, theta, seed=10007 * ep + b)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(loss.item())
            with torch.no_grad():                    # the new readout row: the same local rule as local_rule
                out = model13(x, record=True)
                traces = readout_trace(out["spikes"][-1], model13.beta)
                t_star = out["mem_out"][:, :, NEW].argmax(0)
                pre = traces[t_star, torch.arange(len(y), device=DEVICE)]
                if norm_ref is None:
                    norm_ref = float((pre ** 2).sum(1).mean().clamp_min(1e-6))
                W[NEW] -= (lr_loc / norm_ref) * (local_error(out["logits"], y)[:, None] * pre).mean(0)
        curve.append(epoch_row(model13, ep, losses, name))
    for h in hooks:
        h.remove()
    moved = max(float(((lin.weight.detach() - r).abs() * (1 - m)[:, None]).max()) for lin, r, m in zip(model13.fc, ref, masks))
    check(f"12 {name}: frozen weights did not move", moved == 0.0, f"largest change of a frozen weight {moved:.1e}")
    per_layer = [sum(model13.offsets[l] <= g < model13.offsets[l + 1] for g in frozen) for l in range(L_H)]
    return model13, curve, {"frozen_hidden": len(frozen), "frozen_per_layer": per_layer, "replay": False,
                            "negatives": neg, "theta": theta.tolist(), "lr": CFG["cl"]["lr"], "local_lr": lr_loc,
                            "old_readout_frozen": True}


def ff_locality_check(model13):
    """1) Layer 1's goodness sends no gradient into layer 0's weights (nothing crosses layers).
    2) Layer 0's gradient equals the closed form sum_t surrogate'(U[t] - thr) x[t] * 2 count / (N T^2 B),
    which uses only the current step (nothing flows through time)."""
    wav, _ = next(cl_batches(seed=33))
    x = make_input(wav)
    G = ff_goodness(ff_counts(model13, x)).mean(1)
    g_below = torch.autograd.grad(G[1], model13.fc[0].weight, retain_graph=True, allow_unused=True)[0]
    no_cross = g_below is None or float(g_below.abs().max()) == 0.0
    g_auto = torch.autograd.grad(G[0], model13.fc[0].weight)[0]
    with torch.no_grad():
        T, B, _ = x.shape
        N0 = model13.hidden[0]
        mem, cnt, sg = x.new_zeros(B, N0), x.new_zeros(B, N0), []
        for t in range(T):
            s, mem = lif_step(mem, model13.fc[0](x[t]), model13.beta, model13.thr)
            cnt += s
            sg.append(1.0 / (1.0 + (math.pi * (mem - model13.thr)) ** 2))
        wgt = 2.0 * cnt / (N0 * T_STEPS ** 2 * B)
        g_cf = sum((wgt * sg[t]).T @ x[t] for t in range(T))
    return no_cross, float(F.cosine_similarity(g_auto.flatten(), g_cf.flatten(), dim=0))


In [ ]:
# ---- the shared runner: one continual-learning run, scored on validation and test
TR = CFG["tradeoff"]
f_init = d12 / "new_row_init.pt"
BASIS_INIT = stage_basis("12a_epochs")
saved_init = load_pt(f_init) if f_init.exists() and "12a_epochs" not in FORCE else {}
if saved_init.get("basis") == BASIS_INIT:
    W_NEW0 = saved_init["w"].to(DEVICE)
else:
    W_NEW0, init_info = init_new_row(MODEL)
    save_pt({"w": W_NEW0.cpu(), "basis": BASIS_INIT, **init_info}, f_init)
    log(f"  new readout row initialised: {init_info}")

KEEP = ["acc_13", "old_mean_acc", "hmean", "forgetting_mean_pts", "forgetting_pts", "drift_forgetting_mean_pts",
        "drift_forgetting_pts", "new_word_acc", "new_word_auc", "new_word_auc_vs_class", "old_clips_called_new",
        "new_word_to_unknown", "per_class", "no_decision"]


def hmean(old, new):
    """Harmonic mean of old-class accuracy and new-word accuracy: high only when both are high."""
    return 2.0 * old * new / max(old + new, 1e-12)


def scores(model13, data, base_old):
    s = summarize(*logits_of(model13, data), base_old)
    s["hmean"] = hmean(s["old_mean_acc"], s["new_word_acc"])
    return {k: s[k] for k in KEEP if k in s}


def cl_batches_mix(seed, replay_frac):
    """The pipeline's cl_batches, plus which old clips keep their label. A share replay_frac of the whole batch is
    labelled old clips (each old clip is labelled with probability 2 * replay_frac)."""
    gen = torch.Generator().manual_seed(seed + 7919)
    for wav, y in cl_batches(seed):
        lab = (y != NEW) & (torch.rand(len(y), generator=gen) < 2 * replay_frac).to(y.device)
        yield wav, y, lab


def cl_loss_mix(logits, y, lab):
    """13-way cross-entropy on new-word clips and labelled old clips; unlabelled old clips only keep the new logit
    below the best old logit. replay_frac 0 and 0.5 reproduce the pipeline's no-replay and replay losses."""
    sup = (y == NEW) | lab
    loss = logits.new_zeros(())
    if bool(sup.any()):
        loss = loss + F.cross_entropy(logits[sup], y[sup])
    if bool((~sup).any()):
        z = logits[~sup]
        loss = loss + F.softplus(z[:, NEW] - z[:, :NEW].max(-1).values).mean()
    return loss


def train_cl(frozen, replay_frac=0.0, lr_mult=1.0, epochs=None, seed=0, eval_epochs=(), tag="",
             lwf=0.0, soft=None, keep_logits=False):
    """One run. Returns validation and test scores at the end (and at eval_epochs), the validation curve,
    how far each layer's weights moved, and the largest change of any frozen weight (must be 0).
    lwf > 0 adds distillation: the 12 old logits are kept close to the base network's on every clip of the batch
    (no labels needed). soft = {"weights": per-layer importance vectors, "lam": strength} adds a penalty on how far
    each neuron's incoming weights move, weighted by that neuron's importance (soft freezing).
    keep_logits=True also returns the final validation and test logits (for calibration)."""
    epochs = epochs or E_STAR
    seed_all(CFG["seed"] + 77 + 1000 * seed)
    model13 = expand_readout(MODEL, W_NEW0)
    masks = plasticity_masks(model13, frozen, freeze_old_readout=True)
    hooks = attach_masks(model13, masks)
    ref = [lin.weight.detach().clone() for lin in model13.fc]
    ref_sq = sum(float((r ** 2).sum()) for r in ref[:L_H])
    opt = torch.optim.Adam(model13.parameters(), lr=CFG["cl"]["lr"] * lr_mult, weight_decay=0.0)
    T_lwf = CFG.get("stability", {}).get("lwf_T", 2.0)
    curve, snaps = [], {}
    for ep in range(epochs):
        model13.train()
        losses = []
        for wav, y, lab in cl_batches_mix(1000 + ep + 100003 * seed, replay_frac):
            x = make_input(wav)
            z = model13(x)["logits"]
            loss = cl_loss_mix(z, y, lab)
            if lwf > 0:
                with torch.no_grad():
                    zb = MODEL(x)["logits"]
                loss = loss + lwf * T_lwf ** 2 * F.kl_div(F.log_softmax(z[:, :NEW] / T_lwf, -1), F.softmax(zb / T_lwf, -1),
                                                         reduction="batchmean")
            if soft is not None:
                pen = sum((w[:, None] * (model13.fc[l].weight - ref[l]) ** 2).sum() for l, w in enumerate(soft["weights"]))
                loss = loss + soft["lam"] * pen / ref_sq
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        v = scores(model13, VAL13, BASE_OLD_VAL)
        curve.append({"epoch": ep + 1, "loss": float(np.mean(losses)), **{k: v[k] for k in (
            "drift_forgetting_mean_pts", "forgetting_mean_pts", "old_mean_acc", "new_word_acc", "new_word_auc", "hmean", "acc_13")}})
        if ep + 1 in eval_epochs:
            snaps[ep + 1] = {"val": v, "test": scores(model13, TEST13, BASE_OLD)}
            log(f"    {tag} ep {ep + 1}: val hmean {v['hmean']:.3f}, drift {v['drift_forgetting_mean_pts']:+.1f} | "
                f"test hmean {snaps[ep + 1]['test']['hmean']:.3f}")
    for h in hooks:
        h.remove()
    moved = max(float(((lin.weight.detach() - r).abs() * (1 - m)[:, None]).max()) for lin, r, m in zip(model13.fc, ref, masks))
    rel = [float((model13.fc[l].weight.detach() - ref[l]).norm() / ref[l].norm()) for l in range(L_H)]
    out = {"val": v, "test": snaps[epochs]["test"] if epochs in snaps else scores(model13, TEST13, BASE_OLD),
           "curve": curve, "snaps": {str(k): s for k, s in snaps.items()}, "frozen_moved": moved,
           "weight_change": rel, "frozen_per_layer": [sum(MODEL.offsets[l] <= g < MODEL.offsets[l + 1] for g in frozen)
                                                      for l in range(L_H)],
           "settings": {"frozen": len(frozen), "replay_frac": replay_frac, "lr_mult": lr_mult, "epochs": epochs, "seed": seed,
                        "lwf": lwf, "soft_lam": None if soft is None else soft["lam"]}}
    if keep_logits:
        Zv, Yv = logits_of(model13, VAL13)
        Zt, Yt = logits_of(model13, TEST13)
        out["logits"] = {"val_Z": Zv, "val_Y": Yv, "test_Z": Zt, "test_Y": Yt}
    del model13, opt
    return out


def run_cached(stage, name, fn):
    """Run fn() once and save its result; a later run with the same settings and inputs loads it instead."""
    d = stage_dir(stage) / "runs"
    d.mkdir(exist_ok=True)
    f, basis = d / f"{name}.json", stage_basis(stage)
    if f.exists() and stage not in FORCE:
        r = load_json(f)
        if r.get("basis") == basis:
            return r
    t0 = time.time()
    r = fn()
    r.update(name=name, basis=basis, seconds=round(time.time() - t0))
    save_json(r, f)
    te = r["test"]
    log(f"{stage} {name}: test drift {te['drift_forgetting_mean_pts']:+.2f} | 13-way forgetting {te['forgetting_mean_pts']:+.2f} | "
        f"'{NEW_WORD}' acc {te['new_word_acc']:.3f} AUC {te['new_word_auc']:.3f} | hmean {te['hmean']:.3f} "
        f"(val {r['val']['hmean']:.3f}) | {r['seconds']} s")
    return r


def frozen_ok(stage, runs):
    worst = max((r.get("frozen_moved", 0.0) for r in runs.values()), default=0.0)
    check(f"{stage}: frozen weights never moved, in all {len(runs)} runs", worst == 0.0, f"largest change {worst:.1e}")


def table(runs, keys=("drift_forgetting_mean_pts", "forgetting_mean_pts", "new_word_acc", "new_word_auc", "hmean", "acc_13")):
    print(f"{'run':<34}{'val hmean':>10}" + "".join(f"{k.replace('_mean_pts', '').replace('_', ' '):>18}" for k in keys))
    for n, r in runs.items():
        print(f"{n:<34}{r['val']['hmean']:>10.3f}" + "".join(f"{r['test'][k]:>18.3f}" for k in keys))


ARMS_REG = {                        # name: (frozen-set kind, replay fraction)
    "no_mask": ("none", 0.0), "mi_mask": ("mi", 0.0), "random_mask": ("random", 0.0),
    "random_layer_matched": ("random_layer_matched", 0.0), "fisher_mask": ("fisher", 0.0),
    "layer0_only": ("layer0", 0.0), "mi_plus_layer0": ("mi+layer0", 0.0), "mi_plus_fisher": ("mi+fisher", 0.0),
    "readout_only": ("all", 0.0), "no_mask_replay": ("none", 0.5), "mi_mask_replay": ("mi", 0.5)}
for _name in COMBO_GROUPS:
    ARMS_REG[_name] = ("combo:" + _name, 0.0)


### 12a · How many epochs to learn "zero"

Six approaches (no mask, MI mask, Fisher mask, the within-layer α = 0.5 blend, no mask + full replay, readout only) are trained for 20 epochs and scored on validation and test at epochs 1, 2, 3, 5, 8, 12, 16 and 20. **E\*** is the epoch with the highest mean validation hmean over the six; every later stage trains for E\* epochs.

**Check:** frozen weights never moved; E\* is one of the scored epochs.

In [ ]:
EP = TR["epoch_sweep"]
EPOCH_RUNS = {}
for a in EP["arms"]:
    kind, rf = ARMS_REG[a]
    EPOCH_RUNS[a] = run_cached("12a_epochs", a, lambda kind=kind, rf=rf, a=a: train_cl(
        frozen_set(kind), replay_frac=rf, epochs=EP["max_epochs"], eval_epochs=EP["eval_epochs"], tag=a))
EPOCH_TABLE = {e: {a: r["snaps"][str(e)] for a, r in EPOCH_RUNS.items()} for e in EP["eval_epochs"]}
mean_val = {e: float(np.mean([s["val"]["hmean"] for s in EPOCH_TABLE[e].values()])) for e in EP["eval_epochs"]}
E_STAR = max(EP["eval_epochs"], key=lambda e: (mean_val[e], -e))
if not is_done("12a_epochs"):
    mark_done("12a_epochs", e_star=E_STAR)
log(f"12a_epochs: E* = {E_STAR} (mean validation hmean {mean_val[E_STAR]:.3f})")


In [ ]:
# ---- CHECK 12a: the epoch table
frozen_ok("12a", EPOCH_RUNS)
check("12a E* is one of the scored epochs, chosen on validation", E_STAR in EP["eval_epochs"], f"E* = {E_STAR}")
print(f"{'epoch':>6}{'mean val hmean':>16}   test hmean / test drift per approach")
for e in EP["eval_epochs"]:
    print(f"{e:>6}{mean_val[e]:>16.3f}   " + " | ".join(f"{a} {s['test']['hmean']:.3f}/{s['test']['drift_forgetting_mean_pts']:+.1f}"
                                                  for a, s in EPOCH_TABLE[e].items()))
print("best validation epoch per approach:",
      {a: max(EP["eval_epochs"], key=lambda e: EPOCH_TABLE[e][a]["val"]["hmean"]) for a in EP["arms"]})
if HAVE_PLT:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
    for a in EP["arms"]:
        es = EP["eval_epochs"]
        axes[0].plot(es, [EPOCH_TABLE[e][a]["val"]["hmean"] for e in es], marker="o", label=a)
        axes[1].plot(es, [EPOCH_TABLE[e][a]["val"]["drift_forgetting_mean_pts"] for e in es], marker="o")
        axes[2].plot(es, [EPOCH_TABLE[e][a]["val"]["new_word_acc"] for e in es], marker="o")
    for ax, t in zip(axes, ["validation hmean", "validation drift (pts)", f"validation '{NEW_WORD}' accuracy"]):
        ax.axvline(E_STAR, ls="--", c="grey", lw=0.8)
        ax.set_title(t)
        ax.set_xlabel("new-word epochs")
    axes[0].legend(fontsize=7)
    savefig("12a_epochs")


### 12b · Regular approaches and the α sweep, at E\* epochs

Every approach from the pipeline notebook (no mask, MI, random, layer-matched random, Fisher, layer 0 only, MI + layer 0, MI + Fisher, readout only, replay with and without the MI mask) plus one masked approach per stage-11b group set (11 α values × 2 rank modes), and the SLDA head as a backprop-free reference. **α\*** is the blend with the highest validation hmean; stage 12c uses it.

In [ ]:
ARM_RUNS = {}
for a, (kind, rf) in ARMS_REG.items():
    ARM_RUNS[a] = run_cached("12b_arms", a, lambda kind=kind, rf=rf, a=a: train_cl(frozen_set(kind), replay_frac=rf, tag=a))


def slda_run():
    m = run_slda_arms()["slda_all"][0]
    return {"val": scores(m, VAL13, BASE_OLD_VAL), "test": scores(m, TEST13, BASE_OLD), "frozen_moved": 0.0}


SLDA_RUN = run_cached("12b_arms", "slda_all", slda_run)
BLENDS = {a: r for a, r in ARM_RUNS.items() if a.startswith("combo_")}
BEST_BLEND = max(BLENDS, key=lambda a: BLENDS[a]["val"]["hmean"])
BEST_BLEND_MODE, BEST_ALPHA = COMBO_META[BEST_BLEND]["mode"], COMBO_META[BEST_BLEND]["alpha"]
if not is_done("12b_arms"):
    mark_done("12b_arms", best_blend=BEST_BLEND)
log(f"12b_arms: best blend on validation = {BEST_BLEND} (mode {BEST_BLEND_MODE}, alpha {BEST_ALPHA})")


In [ ]:
# ---- CHECK 12b: results tables
frozen_ok("12b", ARM_RUNS)
check("12b alpha* chosen on validation among the blended masks", BEST_BLEND in BLENDS, BEST_BLEND)
table({**{a: r for a, r in ARM_RUNS.items() if not a.startswith("combo_")}, "slda_all": SLDA_RUN})
print()
table(BLENDS)
if HAVE_PLT:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
    for mode in CFG["combo"]["rank_modes"]:
        rs = sorted((a for a in BLENDS if COMBO_META[a]["mode"] == mode), key=lambda a: COMBO_META[a]["alpha"])
        xs = [COMBO_META[a]["alpha"] for a in rs]
        axes[0].plot(xs, [BLENDS[a]["val"]["hmean"] for a in rs], marker="o", label=mode)
        axes[1].plot(xs, [BLENDS[a]["test"]["drift_forgetting_mean_pts"] for a in rs], marker="o", label=mode)
    axes[0].set_title("validation hmean by alpha")
    axes[1].set_title("test drift by alpha (pts)")
    for ax in axes:
        ax.set_xlabel("alpha (0 = Fisher, 1 = MI)")
        ax.legend()
    savefig("12b_alpha")


### 12c · How many neurons to freeze

For each selection method, a **budget** of 5, 10, 20, 30, 50 or 70% of all hidden neurons is frozen. Neurons are taken **round-robin over the 12 classes**, each class contributing its next most important neuron in turn, so every class gets a fair share and a larger budget always contains the smaller ones. Methods: per-class **MI**; per-class **Fisher** ranked over the whole network (`fisher_global`, depth-biased) or within each layer (`fisher_layer`); the **best blend** from 12b; and a size-matched **random** set. The best (method, budget) on validation hmean is used in 12d and 12e.

In [ ]:
def class_rankings(method):
    """Per class, every hidden neuron from most to least important."""
    wl = BEST_BLEND_MODE == "within_layer"
    if method == "mi":
        S = MI["per_class"].double()
    elif method == "fisher_global":
        S = FISH["naive"].double()
    elif method == "fisher_layer":
        S = rank_scores(FISH["naive"], True)
    elif method == "blend_best":
        S = BEST_ALPHA * rank_scores(MI["per_class"], wl) + (1 - BEST_ALPHA) * rank_scores(FISH["naive"], wl)
    else:
        raise ValueError(method)
    S = S + TIEBREAK
    return [torch.argsort(S[c], descending=True).tolist() for c in range(S.shape[0])]


def budget_set(method, B):
    """B neurons, picked round-robin over classes (nested: a larger budget contains every smaller one)."""
    if method == "random":
        return set(np.random.default_rng(CFG["seed"] + 555).permutation(N_H)[:B].tolist())
    lists, out, r = class_rankings(method), [], 0
    seen = set()
    while len(out) < B:
        for lst in lists:
            if len(out) >= B:
                break
            if lst[r] not in seen:
                seen.add(lst[r])
                out.append(lst[r])
        r += 1
    return set(out)


BUDGETS = {frac: max(1, int(round(frac * N_H))) for frac in TR["budgets"]}
BUDGET_RUNS = {}
for m in TR["budget_methods"]:
    for frac, B in BUDGETS.items():
        name = f"{m}_b{int(round(100 * frac)):02d}"
        BUDGET_RUNS[name] = run_cached("12c_budget", name, lambda m=m, B=B, name=name: {
            **train_cl(budget_set(m, B), tag=name), "method": m, "budget": B, "budget_frac": frac})
selectable = {n: r for n, r in BUDGET_RUNS.items() if r["method"] != "random"}
BEST_MASK = max(selectable, key=lambda n: selectable[n]["val"]["hmean"])
BEST_METHOD, BEST_B = BUDGET_RUNS[BEST_MASK]["method"], BUDGET_RUNS[BEST_MASK]["budget"]
if not is_done("12c_budget"):
    mark_done("12c_budget", best_mask=BEST_MASK)
log(f"12c_budget: best mask on validation = {BEST_MASK} ({BEST_B} neurons)")


In [ ]:
# ---- CHECK 12c: budget curves
frozen_ok("12c", BUDGET_RUNS)
small, big = min(BUDGETS.values()), max(BUDGETS.values())
check("12c budgets are nested (the smallest set sits inside the largest) for every method",
      all(budget_set(m, small) <= budget_set(m, big) for m in TR["budget_methods"]))
check("12c each budget freezes exactly B neurons", all(r["settings"]["frozen"] == r["budget"] for r in BUDGET_RUNS.values()))
table(BUDGET_RUNS)
if HAVE_PLT:
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
    for m in TR["budget_methods"]:
        rs = sorted((r for r in BUDGET_RUNS.values() if r["method"] == m), key=lambda r: r["budget"])
        xs = [100 * r["budget_frac"] for r in rs]
        axes[0].plot(xs, [r["test"]["drift_forgetting_mean_pts"] for r in rs], marker="o", label=m)
        axes[1].plot(xs, [r["test"]["new_word_acc"] for r in rs], marker="o", label=m)
        axes[2].plot(xs, [r["val"]["hmean"] for r in rs], marker="o", label=m)
    for ax, t in zip(axes, ["test drift (pts)", f"test '{NEW_WORD}' accuracy", "validation hmean"]):
        ax.set_title(t)
        ax.set_xlabel("% of hidden neurons frozen")
    axes[0].legend(fontsize=8)
    savefig("12c_budget")


### 12d · Learning rate × replay fraction

The two remaining dials, crossed: the continual-learning **learning rate** (0.25×, 1×, 4×) and the **replay fraction** (0, 1, 5, 25, 50% of each batch as labelled old clips), each with **no mask** and with the **best mask** from 12c. This maps the forgetting–plasticity frontier: every run is a point (old-class accuracy, "zero" accuracy).

In [ ]:
MASKS_12D = {"none": set(), "best": budget_set(BEST_METHOD, BEST_B)}
TRADE_RUNS = {}
for mk, fz in MASKS_12D.items():
    for lm in TR["lr_mults"]:
        for rf in TR["replay_fracs"]:
            name = f"{mk}_lr{lm:g}_rp{rf:g}"
            TRADE_RUNS[name] = run_cached("12d_tradeoff", name, lambda fz=fz, lm=lm, rf=rf, name=name, mk=mk: {
                **train_cl(fz, replay_frac=rf, lr_mult=lm, tag=name), "mask": mk, "lr_mult": lm, "replay_frac": rf})
if not is_done("12d_tradeoff"):
    mark_done("12d_tradeoff")


In [ ]:
# ---- CHECK 12d: the frontier
frozen_ok("12d", TRADE_RUNS)
table(TRADE_RUNS)


def pareto(points):
    """Indices of points not beaten on both coordinates (higher is better on both)."""
    return [i for i, (x, y) in enumerate(points) if not any(x2 >= x and y2 >= y and (x2, y2) != (x, y) for x2, y2 in points)]


ALL_RUNS = {**{f"12b/{k}": v for k, v in ARM_RUNS.items()}, **{f"12c/{k}": v for k, v in BUDGET_RUNS.items()},
            **{f"12d/{k}": v for k, v in TRADE_RUNS.items()}}
names = list(ALL_RUNS)
pts = [(ALL_RUNS[n]["test"]["old_mean_acc"], ALL_RUNS[n]["test"]["new_word_acc"]) for n in names]
FRONT = [names[i] for i in pareto(pts)]
print("\nPareto front (test old-class accuracy vs test new-word accuracy):")
for n in sorted(FRONT, key=lambda n: -ALL_RUNS[n]["test"]["old_mean_acc"]):
    t = ALL_RUNS[n]["test"]
    print(f"  {n:<40} old {t['old_mean_acc']:.3f}  new {t['new_word_acc']:.3f}  hmean {t['hmean']:.3f}")
if HAVE_PLT:
    plt.figure(figsize=(7, 5))
    for grp, mk in [("12b", "o"), ("12c", "s"), ("12d", "^")]:
        sel = [n for n in names if n.startswith(grp)]
        plt.scatter([ALL_RUNS[n]["test"]["old_mean_acc"] for n in sel], [ALL_RUNS[n]["test"]["new_word_acc"] for n in sel],
                    marker=mk, s=18, alpha=0.7, label=grp)
    fr = sorted(FRONT, key=lambda n: ALL_RUNS[n]["test"]["old_mean_acc"])
    plt.plot([ALL_RUNS[n]["test"]["old_mean_acc"] for n in fr], [ALL_RUNS[n]["test"]["new_word_acc"] for n in fr], "k--", lw=0.8)
    plt.axvline(float(BASE_OLD.mean()), c="grey", lw=0.8, ls=":")
    plt.xlabel("old-class accuracy, 13-way (test)")
    plt.ylabel(f"'{NEW_WORD}' accuracy (test)")
    plt.legend()
    plt.title("forgetting vs plasticity (dotted: base network)")
    savefig("12d_frontier")


### 12e · Seeds

The key comparison is repeated with every seed in `CFG["tradeoff"]["seeds"]`. The base network is the same; the seed changes the order of the continual-learning batches and the augmentation (each mask stays the same set of neurons). Reported as mean ± SD.

In [ ]:
KEY = {"no_mask": set(), "mi_mask": frozen_set("mi"), "fisher_mask": frozen_set("fisher"),
       f"best ({BEST_MASK})": budget_set(BEST_METHOD, BEST_B), f"random ({BEST_B})": budget_set("random", BEST_B),
       "no_mask_replay": "replay", "readout_only": frozen_set("all")}
SEED_RUNS = {}
for k, fz in KEY.items():
    for sd in TR["seeds"]:
        name = f"{k.split(' ')[0]}{'_' + str(BEST_B) if k.startswith(('best', 'random')) else ''}_s{sd}"
        rf = 0.5 if fz == "replay" else 0.0
        SEED_RUNS[(k, sd)] = run_cached("12e_seeds", name, lambda fz=fz, rf=rf, sd=sd, name=name: train_cl(
            set() if fz == "replay" else fz, replay_frac=rf, seed=sd, tag=name))
if not is_done("12e_seeds"):
    mark_done("12e_seeds")
SEED_SUMMARY = {}
for k in KEY:
    rs = [SEED_RUNS[(k, sd)]["test"] for sd in TR["seeds"]]
    SEED_SUMMARY[k] = {m: {"mean": float(np.mean([r[m] for r in rs])), "sd": float(np.std([r[m] for r in rs]))}
                       for m in ("drift_forgetting_mean_pts", "forgetting_mean_pts", "new_word_acc", "new_word_auc", "hmean", "acc_13", "old_mean_acc")}


In [ ]:
# ---- CHECK 12e: mean +- SD over seeds
frozen_ok("12e", {f"{k}/{s}": v for (k, s), v in SEED_RUNS.items()})
if len(TR["seeds"]) > 1:
    check("12e different seeds give different runs", len({round(SEED_RUNS[("no_mask", s)]["test"]["drift_forgetting_mean_pts"], 6)
                                                        for s in TR["seeds"]}) > 1)
print(f"{'approach':<28}" + "".join(f"{m:>22}" for m in ("drift", "13-way forgetting", "zero acc", "hmean", "13-way acc")))
for k, s in SEED_SUMMARY.items():
    print(f"{k:<28}" + "".join(f"{s[m]['mean']:>14.3f} ± {s[m]['sd']:<5.3f}" for m in
                               ("drift_forgetting_mean_pts", "forgetting_mean_pts", "new_word_acc", "hmean", "acc_13")))


### 12f · Stability first: keep the old classes, then learn as much "zero" as possible

The selection rule changes here: a setting is **acceptable** only if old-class accuracy on validation falls by at most `max_old_drop_pts` (2 points) below the base network's; among acceptable settings, the one with the highest validation "zero" accuracy wins. Four ways to protect the old classes, each a dial from "learn freely" to "barely move":

- **Calibration** (after training, no retraining): a bias added to the "zero" output, chosen on validation as the highest "zero" accuracy that keeps the old classes within the limit. It removes stealing; it cannot undo drift.
- **Distillation** (Learning without Forgetting): the 12 old logits are kept close to the base network's on every clip of the batch. It needs no labels, so it uses exactly the data the no-replay runs see. Strength λ swept, with no mask and with the best mask from 12c.
- **Soft freezing**: a penalty on how far each neuron's incoming weights move, weighted equally (plain L2), by the neuron's Fisher information, or by its best-blend MI + Fisher score. Strength λ swept, no mask.
- **Freezing 80–95%** of the hidden neurons (best blend and random), to complete the 12c curve.

Every run here keeps its validation and test logits, so each is reported as trained and calibrated. The rule is also applied to every earlier run, as trained.

**Check:** frozen weights never moved; calibration with bias 0 reproduces the trained scores; at least one setting meets the limit.

In [ ]:
ST = CFG["stability"]
TOL = ST["max_old_drop_pts"] / 100.0
BASE_VAL_OLD = float(BASE_OLD_VAL.mean())


def logit_scores(Z, Y, base_old):
    s = summarize(Z, Y, base_old)
    s["hmean"] = hmean(s["old_mean_acc"], s["new_word_acc"])
    return {k: s[k] for k in KEEP if k in s}


def old_new(Z, Y, b):
    """Old-class (mean per class) and new-word accuracy with bias b added to the new output."""
    Zb = Z.clone()
    Zb[:, NEW] += b
    pc, _ = class_scores(decide(Zb), Y, 13)
    return float(pc[:NEW].mean()), float(pc[NEW])


def calibrate(L):
    """Bias on the new output, chosen on validation: the highest new-word accuracy whose old-class accuracy stays within
    TOL of the base network; if no bias gets there, the bias with the highest old-class accuracy."""
    Zv, Yv, Zt, Yt = L["val_Z"], L["val_Y"], L["test_Z"], L["test_Y"]
    g = (Zv[:, NEW] - Zv[:, :NEW].max(1).values).float()
    qs = torch.quantile(g, torch.linspace(0, 1, 201))
    biases = sorted({0.0} | {float(-q + d) for q in qs for d in (-1e-4, 1e-4)})
    rows = [(b, *old_new(Zv, Yv, b)) for b in biases]
    ok = [r for r in rows if r[1] >= BASE_VAL_OLD - TOL]
    b = max(ok, key=lambda r: (r[2], r[1]))[0] if ok else max(rows, key=lambda r: (r[1], r[2]))[0]
    Zv2, Zt2 = Zv.clone(), Zt.clone()
    Zv2[:, NEW] += b
    Zt2[:, NEW] += b
    return {"bias": b, "feasible": bool(ok), "val": logit_scores(Zv2, Yv, BASE_OLD_VAL), "test": logit_scores(Zt2, Yt, BASE_OLD),
            "val_curve": [[r[0], r[1], r[2]] for r in rows[::4]]}


def stab_run(name, frozen, stage="12f_stability", seed=0, **kw):
    """One run that keeps its logits and is also scored after calibration."""
    def fn():
        r = train_cl(frozen, keep_logits=True, tag=name, seed=seed, **kw)
        L = r.pop("logits")
        save_pt(L, stage_dir(stage) / "runs" / f"{name}_logits.pt")
        r["calibrated"] = calibrate(L)
        return r
    return run_cached(stage, name, fn)


def importance(kind):
    """Per-neuron importance for soft freezing, scaled to mean 1 over all hidden neurons."""
    if kind == "uniform":
        v = torch.ones(N_H, dtype=torch.float64)
    elif kind == "fisher":
        v = FISH["naive"].double().mean(0)
    elif kind == "blend":
        wl = BEST_BLEND_MODE == "within_layer"
        v = (BEST_ALPHA * rank_scores(MI["per_class"], wl) + (1 - BEST_ALPHA) * rank_scores(FISH["naive"], wl)).max(0).values
    else:
        raise ValueError(kind)
    v = v / v.mean()
    return [v[MODEL.offsets[l]:MODEL.offsets[l + 1]].float().to(DEVICE) for l in range(L_H)]


BEST_SET = budget_set(BEST_METHOD, BEST_B)
STAB_CFGS = {"ref_readout_only": (frozen_set("all"), {}), "ref_no_mask": (set(), {}), "ref_mi_mask": (frozen_set("mi"), {}),
             "ref_best_blend": (frozen_set("combo:" + BEST_BLEND), {}), "ref_best_mask": (BEST_SET, {}),
             "ref_replay_25": (set(), {"replay_frac": 0.25})}           # name: (frozen neurons, train_cl options)
for mk, fz in [("none", set()), ("best", BEST_SET)]:
    for lam in ST["lwf_lambdas"]:
        STAB_CFGS[f"lwf_{mk}_{lam:g}"] = (fz, {"lwf": lam})
for kind in ST["soft_importance"]:
    imp = importance(kind)
    for lam in ST["soft_lambdas"]:
        STAB_CFGS[f"soft_{kind}_{lam:g}"] = (set(), {"soft": {"weights": imp, "lam": lam}})
for m in ST["extreme_methods"]:
    for frac in ST["extreme_budgets"]:
        STAB_CFGS[f"freeze_{m}_b{int(round(100 * frac)):02d}"] = (budget_set(m, max(1, int(round(frac * N_H)))), {})
STAB_RUNS = {name: stab_run(name, fz, **kw) for name, (fz, kw) in STAB_CFGS.items()}


In [ ]:
# ---- CHECK 12f: the stability-first selection
frozen_ok("12f", STAB_RUNS)
L0 = load_pt(stage_dir("12f_stability") / "runs" / "ref_no_mask_logits.pt")
o0, n0 = old_new(L0["test_Z"], L0["test_Y"], 0.0)
check("12f calibration with bias 0 reproduces the trained test scores",
      abs(o0 - STAB_RUNS["ref_no_mask"]["test"]["old_mean_acc"]) < 1e-6 and abs(n0 - STAB_RUNS["ref_no_mask"]["test"]["new_word_acc"]) < 1e-6,
      f"old {o0:.4f}, new {n0:.4f}")
print(f"base network old-class accuracy: {BASE_VAL_OLD:.3f} (validation), {float(BASE_OLD.mean()):.3f} (test); "
      f"limit: at most {ST['max_old_drop_pts']:g} points lower on validation\n")
print(f"{'run':<26}{'old':>8}{'zero':>8}{'all 13':>9}{'drift':>8}   |  calibrated: {'bias':>7}{'ok':>4}{'old':>8}{'zero':>8}{'all 13':>9}   (test)")
for n, r in STAB_RUNS.items():
    t, c = r["test"], r["calibrated"]
    print(f"{n:<26}{t['old_mean_acc']:>8.3f}{t['new_word_acc']:>8.3f}{t['acc_13']:>9.3f}{t['drift_forgetting_mean_pts']:>8.1f}   |  "
          f"{'':>12}{c['bias']:>7.2f}{'yes' if c['feasible'] else 'no':>4}{c['test']['old_mean_acc']:>8.3f}"
          f"{c['test']['new_word_acc']:>8.3f}{c['test']['acc_13']:>9.3f}")

CANDIDATES = {}
for grp, runs in [("12b", ARM_RUNS), ("12c", BUDGET_RUNS), ("12d", TRADE_RUNS), ("12f", STAB_RUNS)]:
    for n, r in runs.items():
        CANDIDATES[f"{grp}/{n}"] = {"val": r["val"], "test": r["test"]}
for n, r in STAB_RUNS.items():
    CANDIDATES[f"12f/{n} (calibrated)"] = {"val": r["calibrated"]["val"], "test": r["calibrated"]["test"]}
CANDIDATES["12b/slda_all"] = {"val": SLDA_RUN["val"], "test": SLDA_RUN["test"]}
feasible = {n: c for n, c in CANDIDATES.items() if c["val"]["old_mean_acc"] >= BASE_VAL_OLD - TOL}
ranked = sorted(feasible, key=lambda n: (-feasible[n]["val"]["new_word_acc"], -feasible[n]["val"]["old_mean_acc"]))
print(f"\n{len(feasible)} of {len(CANDIDATES)} settings keep validation old-class accuracy within {ST['max_old_drop_pts']:g} points; "
      f"best by validation '{NEW_WORD}' accuracy:")
for n in ranked[:15]:
    t = feasible[n]["test"]
    print(f"  {n:<44} test: old {t['old_mean_acc']:.3f} (drop {100 * (float(BASE_OLD.mean()) - t['old_mean_acc']):+.1f} pts), "
          f"'{NEW_WORD}' {t['new_word_acc']:.3f}, all 13 {t['acc_13']:.3f}")
STAB_BEST = ranked[0] if ranked else None
check("12f at least one setting meets the stability limit on validation", bool(ranked), STAB_BEST or "none")
STAB_SUMMARY = {"tolerance_pts": ST["max_old_drop_pts"], "base_val_old": BASE_VAL_OLD, "base_test_old": float(BASE_OLD.mean()),
                "best": STAB_BEST, "best_test": feasible[STAB_BEST]["test"] if STAB_BEST else None,
                "feasible_ranked": [{"name": n, "val": feasible[n]["val"], "test": feasible[n]["test"]} for n in ranked[:30]],
                "runs": {n: {"test": r["test"], "val": r["val"], "settings": r.get("settings"), "weight_change": r.get("weight_change"),
                             "calibrated": r["calibrated"]} for n, r in STAB_RUNS.items()}}
if not is_done("12f_stability"):
    mark_done("12f_stability")
log(f"12f_stability: best setting within the limit = {STAB_BEST}")
if HAVE_PLT:
    plt.figure(figsize=(7, 5))
    for n, r in STAB_RUNS.items():
        t, c = r["test"], r["calibrated"]["test"]
        plt.plot([t["old_mean_acc"], c["old_mean_acc"]], [t["new_word_acc"], c["new_word_acc"]], c="lightgrey", lw=0.8)
        plt.scatter(t["old_mean_acc"], t["new_word_acc"], s=14, c="grey")
        plt.scatter(c["old_mean_acc"], c["new_word_acc"], s=14, c="tab:blue")
    plt.axvline(float(BASE_OLD.mean()), c="k", lw=0.8, ls=":")
    plt.axvline(float(BASE_OLD.mean()) - TOL, c="tab:red", lw=0.8, ls="--")
    plt.xlabel("old-class accuracy (test)")
    plt.ylabel(f"'{NEW_WORD}' accuracy (test)")
    plt.title("12f: as trained (grey) and calibrated (blue); dashed = stability limit")
    savefig("12f_stability")


### 12g · Seeds for the stability-first winners

The top settings of the 12f ranking (the `top_k` best runs that meet the limit on validation, as trained or calibrated), plus readout-only and 25% replay as references, repeated with the seeds in `CFG["stability_seeds"]["seeds"]`. Seed 0 is the 12f run itself. Each run is calibrated on its own validation logits, exactly as in 12f. Reported as mean ± SD, as trained and calibrated, with the test drop in old-class accuracy against the base network.

In [ ]:
SS = CFG["stability_seeds"]
TOP12F = []
for n in ranked:
    if n.startswith("12f/"):
        b = n[len("12f/"):].replace(" (calibrated)", "")
        if b not in TOP12F:
            TOP12F.append(b)
    if len(TOP12F) >= SS["top_k"]:
        break
for ref in SS["references"]:
    if ref not in TOP12F:
        TOP12F.append(ref)
log(f"12g: seeds {SS['seeds']} for {TOP12F}")
SEED12G = {}
for name in TOP12F:
    fz, kw = STAB_CFGS[name]
    SEED12G[name] = {0: STAB_RUNS[name]}
    for sd in SS["seeds"]:
        SEED12G[name][sd] = stab_run(f"{name}_s{sd}", fz, stage="12g_stability_seeds", seed=sd, **kw)


In [ ]:
# ---- CHECK 12g: mean +- SD over seeds, as trained and calibrated
frozen_ok("12g", {f"{n}/{s}": r for n, runs in SEED12G.items() for s, r in runs.items() if s != 0})
if SS["seeds"]:
    n0 = TOP12F[0]
    check("12g different seeds give different runs",
          len({round(r["test"]["old_mean_acc"], 6) for r in SEED12G[n0].values()}) > 1, n0)


def ms(vals):
    return {"mean": float(np.mean(vals)), "sd": float(np.std(vals))}


base_old_test = float(BASE_OLD.mean())
SEED12G_SUMMARY = {}
for name, runs in SEED12G.items():
    rs = list(runs.values())
    SEED12G_SUMMARY[name] = {
        "seeds": sorted(runs),
        "trained": {m: ms([r["test"][m] for r in rs]) for m in ("old_mean_acc", "new_word_acc", "acc_13", "drift_forgetting_mean_pts")},
        "calibrated": {m: ms([r["calibrated"]["test"][m] for r in rs]) for m in ("old_mean_acc", "new_word_acc", "acc_13")},
        "old_drop_pts_calibrated": ms([100 * (base_old_test - r["calibrated"]["test"]["old_mean_acc"]) for r in rs]),
        "feasible_on_val": int(sum(bool(r["calibrated"]["feasible"]) for r in rs)), "n": len(rs)}
print(f"base network old-class accuracy (test) {base_old_test:.3f}\n")
print(f"{'setting':<22}{'n':>3}   {'as trained: old / zero / all 13':<34}   {'calibrated: old / zero / all 13':<34}{'old drop (pts)':>16}")
for name, s in SEED12G_SUMMARY.items():
    tr, ca = s["trained"], s["calibrated"]
    f = lambda d: f"{d['mean']:.3f}±{d['sd']:.3f}"
    print(f"{name:<22}{s['n']:>3}   {f(tr['old_mean_acc'])} {f(tr['new_word_acc'])} {f(tr['acc_13'])}   "
          f"{f(ca['old_mean_acc'])} {f(ca['new_word_acc'])} {f(ca['acc_13'])}"
          f"{s['old_drop_pts_calibrated']['mean']:>9.2f} ± {s['old_drop_pts_calibrated']['sd']:.2f}")
if not is_done("12g_stability_seeds"):
    mark_done("12g_stability_seeds")


## 13 · Summary

All key numbers in `RUN_DIR/summary.json`, with the list of failed checks; the figures are in `RUN_DIR/plots/`.

In [ ]:
def slim(r):
    return {k: r[k] for k in ("val", "test", "frozen_per_layer", "weight_change", "settings", "method", "budget", "budget_frac",
                              "mask", "lr_mult", "replay_frac", "curve") if k in r}


hist = load_json(d7 / "history.json")
SUMMARY = {
    "run": load_json(RUN / "config.json"),
    "training": {"epochs_run": len(hist), "best_epoch": BASE_EVAL["best_epoch"],
                 "validation_acc": BASE_EVAL["validation"]["acc"], "test_acc": BASE_EVAL["testing"]["acc"],
                 "validation_loss": BASE_EVAL["validation"].get("loss"),
                 "test_per_class": dict(zip(CLASSES_12, BASE_EVAL["testing"]["per_class"])),
                 "history": [{k: r.get(k) for k in ("epoch", "loss", "train_acc", "val_loss", "val_acc", "lr")} for r in hist]},
    "fisher": load_json(d8 / "fisher_summary.json"),
    "mi": load_json(d9 / "mi_summary.json"),
    "groups": {**STATS, "mi_union_size": len(MI_UNION)},
    "ablation": {m: SPEC[m] for m in SPEC},
    "combo": {k: {kk: vv for kk, vv in v.items() if kk != "rows"} for k, v in COMBO.items()},
    "epochs": {"e_star": E_STAR, "mean_val_hmean": mean_val,
               "table": {str(e): {a: {"val": s["val"], "test": s["test"]} for a, s in row.items()} for e, row in EPOCH_TABLE.items()}},
    "arms": {**{k: slim(v) for k, v in ARM_RUNS.items()}, "slda_all": slim(SLDA_RUN)},
    "best_blend": {"name": BEST_BLEND, "mode": BEST_BLEND_MODE, "alpha": BEST_ALPHA},
    "budget": {k: slim(v) for k, v in BUDGET_RUNS.items()},
    "best_mask": {"name": BEST_MASK, "method": BEST_METHOD, "neurons": BEST_B},
    "tradeoff": {k: slim(v) for k, v in TRADE_RUNS.items()},
    "pareto_front": FRONT,
    "seeds": SEED_SUMMARY,
    "stability": STAB_SUMMARY,
    "stability_seeds": SEED12G_SUMMARY,
    "base_old_mean_acc": float(BASE_OLD.mean()),
    "checks": {"total": len(CHECKS), "failed": [k for k, v in CHECKS.items() if not v["passed"]],
               "skipped": [k for k, v in CHECKS.items() if v.get("skipped")]},
}
save_json(SUMMARY, RUN / "summary.json")
log("=" * 72)
log(f"summary written to {(RUN / 'summary.json').resolve()}")
log(f"base network: {len(hist)} epochs, best (lowest validation loss) at epoch {BASE_EVAL['best_epoch']} | "
    f"validation {BASE_EVAL['validation']['acc']:.4f} | test {BASE_EVAL['testing']['acc']:.4f}")
log(f"E* = {E_STAR} | best blend {BEST_BLEND} | best mask {BEST_MASK} ({BEST_B} neurons)")
log(f"checks: {len(CHECKS)} run, {len(SUMMARY['checks']['failed'])} failed, {len(SUMMARY['checks']['skipped'])} skipped"
    + (": " + "; ".join(SUMMARY["checks"]["failed"]) if SUMMARY["checks"]["failed"] else ""))
